In [29]:
import json
import os
import csv
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace, BNode, Dataset
from rdflib.namespace import SKOS, DCTERMS, DCMITYPE, RDF, RDFS, XSD, PROV, SDO, TIME, FOAF, split_uri
from rdflib.compare import isomorphic
from SPARQLWrapper import SPARQLWrapper, CSV
from openai import OpenAI
import ollama
from uk2us import uk_us

import requests
import xml.etree.ElementTree as ET
from html.parser import HTMLParser
from html import unescape
import ftfy
import re
from keybert import KeyBERT

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
from collections import defaultdict, Counter

import lemminflect
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import inflect

import itertools
import pandas as pd
from urllib.parse import urlparse
from bert_score import score
import time
from pathlib import Path
import importlib
from tempfile import NamedTemporaryFile

In [11]:
# Opening config file, the config structure is:
# {"access_token_zenodo":"......"
# "openai_api_key":"......"}

config = open('config', 'r')
config = json.load(config)

os.environ['OPENAI_API_KEY'] = config['openai_api_key']
os.environ['OLLAMA_API_KEY'] = config['ollama_api_key']

In [4]:
# Load the JSON dataset
with open('benchmarks/text_RDF_gs.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Access the dataset
dataset = data['dataset']

In [5]:
def load_graph(data):
    g = Graph()
    g.parse(data=data, format="turtle")
    return g

In [6]:
def print_rdf(rdf):
    g = Graph()
    g.parse(data=rdf, format="turtle")

    for s, p, o in g:
        print(s, p, o)

In [7]:
# Namespaces
she = Namespace("https://soilwise-he.github.io/soil-health#")
agrovoc = Namespace("http://aims.fao.org/aos/agrovoc/")
agrontology = Namespace("http://aims.fao.org/aos/agrontology#")
sio = Namespace("http://semanticscience.org/resource/")
glosis_lh = Namespace("http://w3id.org/glosis/model/layerhorizon/")
glosis_sp = Namespace("http://w3id.org/glosis/model/siteplot/")
qudt = Namespace("http://qudt.org/schema/qudt/")
unit = Namespace("http://qudt.org/vocab/unit/")
iso11074 = Namespace("https://data.geoscience.earth/ncl/ISO11074v2025/")
obo = Namespace("http://purl.obolibrary.org/obo/")
wdt = Namespace("http://www.wikidata.org/prop/direct/")
biolink = Namespace("https://w3id.org/biolink/vocab/")
afox = Namespace("http://purl.allotrope.org/ontologies/property#")
afor = Namespace("http://purl.allotrope.org/ontologies/result#")
sorelsc = Namespace("http://sweetontology.net/relaSci/")
sorelpr = Namespace("http://sweetontology.net/relaProvenance/")
sohuj = Namespace("http://sweetontology.net/humanJurisdiction/")
sorelph = Namespace("http://sweetontology.net/relaPhysical/")
sorelm = Namespace("http://sweetontology.net/relaMath/")
sorepsg = Namespace("http://sweetontology.net/reprSpaceGeometry/")
bao = Namespace("http://www.bioassayontology.org/bao#")
repr = Namespace("https://w3id.org/reproduceme#")
sorelch = Namespace("http://sweetontology.net/relaChemical/")
sorelsp = Namespace("http://sweetontology.net/relaSpace/")
om = Namespace("http://www.ontology-of-units-of-measure.org/resource/om-2/")
gemet = Namespace("http://www.eionet.europa.eu/gemet/concept/")
inrae = Namespace("http://opendata.inrae.fr/thesaurusINRAE/")

### Convert text to RDF triples using LLMs

In [8]:
# System prompt for zero-shot learning
system_prompt_text = """# RDF Knowledge Graph Instructions for Constructing Soil Health Knowledge Graph

## 1. Overview

You are a top-tier algorithm designed for extracting information from unstructured text to build an RDF knowledge graph in the domain of **soil science and soil health**.
Your task is to identify entities, relationships and references from a given text, which will be a **single paragraph from a soil health report**.
Try to capture as much information from the text as possible without sacrificing accuracy. Do not add any information that is not explicitly mentioned in the text.

## 2. Subjects

- **Subjects** represent entities and concepts relevant to **soil science and soil health**.
- Keep subjects atomic and distinct; split combined mentions into separate subjects and do not include conjunctions such as "and" inside a single subject.
- Attempt to extract as many entities and concepts as you can within this domain.
- The aim is to achieve simplicity and clarity in the knowledge graph, making it accessible for a vast audience.

## 3. Labeling Subjects

- **Consistency**: Ensure you assign at least one available class from existing ontologies to each subject. **Do not customize classes**.
  - When you identify a subject as a concept, always use `'skos:Concept'` to label it.
  - If a subject is not a concept, please search for a class from an extant ontology to label it.
  - For every entity and concept, include one `skos:prefLabel` with the human-readable name in lowercase found in the text.
- **URIs**: URIs should be names or human-readable identifiers derived from the text. Use `she: <https://soilwise-he.github.io/soil-health#>` as prefix. Split words in URIs by capitalizing initial letters.
   
## 4. Relations and Predicates

- **Relationships** represent connections between entities.
- Attempt to extract as many relations as you can.
- Ensure consistency and generality in relationship predicates when constructing knowledge graphs. Instead of using specific and momentary predicates, **make sure to use general and timeless relationship predicates.**

## 5. Nested Triples with Blank Nodes

- **Blank Nodes**: When a relationship requires **multiple contextual qualifiers** (e.g., different **conditions** or **locations**), use blank nodes to create nested triples.
- Use blank nodes only when a relationship requires multiple properties and a single predicate-object pair cannot fully capture the text's nuance.

## 6. Coreference Resolution

- **Maintain Entity/Concept Consistency**: When extracting entities and concepts, it's vital to ensure consistency.
- If an entity or concept is mentioned multiple times in the text but referred to by different forms, always use the most complete identifier and preferred label for that entity or concept throughout the knowledge graph.

## 7. Objects

- **Retaining Text as Objects**: Do not retain the given text as objects, except in the following case:
  - **Definitions**: If the text is defining concepts, include the full definition as an object and use `'skos:definition'` as the predicate.
- **Important**: Even when you retain parts of the text as objects (definitions), you must still extract entities, relationships, and references according to the previous instructions.

## 8. References and Citations

- **Extract References**: Identify any references or citations that appear in the text.
- **Labeling References**:
  - Use Dublin Core (`dcterms`) to label references.
  - Represent each reference as a `dcterms:BibliographicResource`.
  - Do not include `skos:prefLabel` for any references, instead use properties from Dublin Core.
- **Connecting Information to References**:
  - Use `dcterms:source` to indicate which information in the text references this literature.
  - When an extracted piece of information is supported by a reference, link it to the corresponding `dcterms:BibliographicResource` using `dcterms:source`.

## 9. Strict Compliance
Adhere to the rules strictly. Non-compliance will result in termination."""

In [9]:
prompt_0 = f"""Please help me convert this text to an RDF knowledge graph:
'{dataset[104]['text']}'
Please think step by step, and only the final result should be output, i.e., the pure RDF knowledge graph serialized in Turtle without any comments. Thank you!"""

In [12]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-6.1-terra",
    messages=[
        {"role": "system", "content": system_prompt_text},
        {"role": "user", "content": prompt_0}
    ]
)

print(completion.choices[0].message.content)

@prefix she: <https://soilwise-he.github.io/soil-health#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

she:EasternEurope a dcterms:Location ;
    skos:prefLabel "eastern europe" ;
    she:hasSoils she:EasternEuropeanSoilTypes ;
    she:hasClimate she:ContinentalClimate, she:MaritimeClimate ;
    she:hasRegion she:SteppicRegion, she:North, she:SouthEast .

she:EasternEuropeanSoilTypes a skos:Concept ;
    skos:prefLabel "soil types" ;
    she:influencedBy she:ContinentalClimate, she:MaritimeClimate .

she:ContinentalClimate a skos:Concept ;
    skos:prefLabel "continental climate" .

she:MaritimeClimate a skos:Concept ;
    skos:prefLabel "maritime climate" .

she:SteppicRegion a dcterms:Location ;
    skos:prefLabel "steppic region" ;
    she:hasTypicalSoilType she:Chernozems, she:Phaeozems, she:Kastanozems .

she:Chernozems a skos:Concept ;
    skos:prefLabel "chern

In [13]:
rdf_data_llm_0 = completion.choices[0].message.content.replace('```turtle', '')
rdf_data_llm_0 = rdf_data_llm_0.replace('```', '')

print_rdf(rdf_data_llm_0)
g_0 = load_graph(rdf_data_llm_0)

https://soilwise-he.github.io/soil-health#PhaeozemChernozemKastanozemSequence https://soilwise-he.github.io/soil-health#hasCharacteristic https://soilwise-he.github.io/soil-health#DarkColors
https://soilwise-he.github.io/soil-health#HighAccumulationOfOrganicMatter https://soilwise-he.github.io/soil-health#involves https://soilwise-he.github.io/soil-health#OrganicMatter
https://soilwise-he.github.io/soil-health#ContinentalClimate http://www.w3.org/2004/02/skos/core#prefLabel continental climate
https://soilwise-he.github.io/soil-health#SouthEast https://soilwise-he.github.io/soil-health#hasClimate https://soilwise-he.github.io/soil-health#SemiAridClimate
https://soilwise-he.github.io/soil-health#Lakes http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#SurfaceWaterlogging http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#Ph

#### Validate Turtle syntax

In [14]:
system_prompt_ttl_checker = """**Task Description**:
You are an **RDF Turtle syntax checker and reviser**. Your task is to:

1. **Receive RDF triples** serialized in Turtle format, which contain syntax errors.
2. Be aware that the input may use the custom prefix: `she: <https://soilwise-he.github.io/soil-health#> .`
3. **Analyze the input** and correct any syntactical issues while preserving the original semantic meaning as much as possible.
4. **Describe the error pattern**: Provide a concise and precise description of the syntax error, similar to the brief error messages seen in code debugging tools.
5. **Fix the error**: Correct the syntax error to produce valid RDF Turtle output.

**Guidelines**:

- Ensure the corrected RDF Turtle is syntactically correct and can be directly parsed by the Python library **"rdflib"**.
- Do not include any additional commentary, code block markers, or language identifiers (e.g., ```turtle) in your output—only the corrected RDF triples."""

In [15]:
prompt_ttl = f"""Please check and correct the following RDF triples:
{completion.choices[0].message.content}"""

In [16]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-6-luna",
    messages=[
        {"role": "system", "content": system_prompt_ttl_checker},
        {"role": "user", "content": prompt_ttl}
    ],
    response_format={ 
        "type": "json_schema", 
        "json_schema": {
            "name": "rdf_turtle_checker_reviser",
            "schema": {
                "type": "object",
                "properties": {
                    "turtle_error_pattern": {
                        "type": "string",
                        "description": "A brief and clear description of the syntax error identified."
                        },
                    "output_rdf": {
                        "type": "string",
                        "description": "The corrected RDF Turtle output without syntax errors."
                        }
                    },
                "required": [
                    "turtle_error_pattern",
                    "output_rdf"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)

print(completion.choices[0].message.content)

{"turtle_error_pattern":"No syntax errors detected; the Turtle input is valid.","output_rdf":"@prefix she: <https://soilwise-he.github.io/soil-health#> .\n@prefix skos: <http://www.w3.org/2004/02/skos/core#> .\n@prefix dcterms: <http://purl.org/dc/terms/> .\n@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n\nshe:EasternEurope a dcterms:Location ;\n    skos:prefLabel \"eastern europe\" ;\n    she:hasSoils she:EasternEuropeanSoilTypes ;\n    she:hasClimate she:ContinentalClimate, she:MaritimeClimate ;\n    she:hasRegion she:SteppicRegion, she:North, she:SouthEast .\n\nshe:EasternEuropeanSoilTypes a skos:Concept ;\n    skos:prefLabel \"soil types\" ;\n    she:influencedBy she:ContinentalClimate, she:MaritimeClimate .\n\nshe:ContinentalClimate a skos:Concept ;\n    skos:prefLabel \"continental climate\" .\n\nshe:MaritimeClimate a skos:Concept ;\n    skos:prefLabel \"maritime climate\" .\n\nshe:SteppicRegion a dcterms:Location ;\n    skos:prefLabel \"steppic region\" ;\n    she

In [17]:
print(json.loads(completion.choices[0].message.content)["turtle_error_pattern"])

No syntax errors detected; the Turtle input is valid.


In [18]:
print(json.loads(completion.choices[0].message.content)["output_rdf"])

@prefix she: <https://soilwise-he.github.io/soil-health#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

she:EasternEurope a dcterms:Location ;
    skos:prefLabel "eastern europe" ;
    she:hasSoils she:EasternEuropeanSoilTypes ;
    she:hasClimate she:ContinentalClimate, she:MaritimeClimate ;
    she:hasRegion she:SteppicRegion, she:North, she:SouthEast .

she:EasternEuropeanSoilTypes a skos:Concept ;
    skos:prefLabel "soil types" ;
    she:influencedBy she:ContinentalClimate, she:MaritimeClimate .

she:ContinentalClimate a skos:Concept ;
    skos:prefLabel "continental climate" .

she:MaritimeClimate a skos:Concept ;
    skos:prefLabel "maritime climate" .

she:SteppicRegion a dcterms:Location ;
    skos:prefLabel "steppic region" ;
    she:hasTypicalSoilType she:Chernozems, she:Phaeozems, she:Kastanozems .

she:Chernozems a skos:Concept ;
    skos:prefLabel "chern

In [19]:
print_rdf(json.loads(completion.choices[0].message.content)["output_rdf"])
g_0 = load_graph(json.loads(completion.choices[0].message.content)["output_rdf"])

https://soilwise-he.github.io/soil-health#PhaeozemChernozemKastanozemSequence https://soilwise-he.github.io/soil-health#hasCharacteristic https://soilwise-he.github.io/soil-health#DarkColors
https://soilwise-he.github.io/soil-health#HighAccumulationOfOrganicMatter https://soilwise-he.github.io/soil-health#involves https://soilwise-he.github.io/soil-health#OrganicMatter
https://soilwise-he.github.io/soil-health#ContinentalClimate http://www.w3.org/2004/02/skos/core#prefLabel continental climate
https://soilwise-he.github.io/soil-health#SouthEast https://soilwise-he.github.io/soil-health#hasClimate https://soilwise-he.github.io/soil-health#SemiAridClimate
https://soilwise-he.github.io/soil-health#Lakes http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#SurfaceWaterlogging http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#Ph

#### Literals re-extractor

In [20]:
system_prompt_extractor = """### Text Re-Extractor for RDF Knowledge Graphs

You are a highly specialized algorithm designed to verify and enhance RDF knowledge graphs generated from unstructured soil science text. Your task is to **ensure completeness and sufficiency** in entity and relationship extraction. This process involves analyzing existing RDF triples, identifying unextracted entities and relationships, and incorporating them into an improved RDF knowledge graph.

**Instructions**:

1. **Input**:
   You will be provided with **LLM-generated RDF triples**: An initial RDF knowledge graph in Turtle format.
2. **Goal**:
   Verify whether the text stored in the RDF (e.g., under `skos:definition` or other literal fields) still contains **entities** or **relationships** that have not been sufficiently extracted. If the text mentions relevant soil science concepts or relationships that are missing in the RDF, you must extract them and include them as new triples.
3. **Entity Extraction**:
   
   - **Entities** refer to all concepts, objects, or elements relevant to soil science and soil health.
   - Entities that are recognized as concepts should be labeled using the **SKOS**. Specifically:
     - Use `skos:Concept` to label concepts.
     - Use `skos:prefLabel` to provide a human-readable name for each concept.
   - If the text refers to **composite entities**, ensure all components are separately identified.
4. **Relationship Extraction**:
   
   - Relationships (predicates) must describe general and universal connections between entities.
   - Avoid specific or temporary predicates. Focus on predicates that ensure **clarity, simplicity, and universality**.
5. **Improvement Rules**:
   
   - For each literal string (e.g., `skos:definition` or text objects), perform a **re-extraction pass**:
     - Check if additional entities or relationships can be identified.
     - If so, extract them and integrate them into new RDF triples.
   - If no new information can be extracted, retain the RDF as-is.
6. **Output**:
   Provide the **improved RDF knowledge graph** in Turtle format without any additional commentary, code block markers, or language identifiers (e.g., ```turtle) in your output. Ensure the graph is clean, well-structured, and adheres to the original namespace conventions."""

In [21]:
prompt_extractor = f"""Here is the LLM-generated RDF triples in Turtle format that need to be processed to extract any additional entities and relationships:
{completion.choices[0].message.content}"""

In [22]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-6-luna",
    messages=[
        {"role": "system", "content": system_prompt_extractor},
        {"role": "user", "content": prompt_extractor}
    ]
)

print(completion.choices[0].message.content)

@prefix she: <https://soilwise-he.github.io/soil-health#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

she:EasternEurope a dcterms:Location ;
    skos:prefLabel "eastern europe" ;
    she:hasSoils she:EasternEuropeanSoilTypes ;
    she:hasClimate she:ContinentalClimate, she:MaritimeClimate ;
    she:hasRegion she:SteppicRegion, she:North, she:SouthEast .

she:EasternEuropeanSoilTypes a skos:Concept ;
    skos:prefLabel "soil types" ;
    she:influencedBy she:ContinentalClimate, she:MaritimeClimate .

she:ContinentalClimate a skos:Concept ;
    skos:prefLabel "continental climate" .

she:MaritimeClimate a skos:Concept ;
    skos:prefLabel "maritime climate" .

she:SteppicRegion a dcterms:Location ;
    skos:prefLabel "steppic region" ;
    she:hasTypicalSoilType she:Chernozems, she:Phaeozems, she:Kastanozems .

she:Chernozems a skos:Concept ;
    skos:prefLabel "chern

In [23]:
rdf_data_llm_0 = completion.choices[0].message.content

print_rdf(rdf_data_llm_0)
g_0 = load_graph(rdf_data_llm_0)

https://soilwise-he.github.io/soil-health#PhaeozemChernozemKastanozemSequence https://soilwise-he.github.io/soil-health#hasCharacteristic https://soilwise-he.github.io/soil-health#DarkColors
https://soilwise-he.github.io/soil-health#HighAccumulationOfOrganicMatter https://soilwise-he.github.io/soil-health#involves https://soilwise-he.github.io/soil-health#OrganicMatter
https://soilwise-he.github.io/soil-health#ContinentalClimate http://www.w3.org/2004/02/skos/core#prefLabel continental climate
https://soilwise-he.github.io/soil-health#SouthEast https://soilwise-he.github.io/soil-health#hasClimate https://soilwise-he.github.io/soil-health#SemiAridClimate
https://soilwise-he.github.io/soil-health#Lakes http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#SurfaceWaterlogging http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2004/02/skos/core#Concept
https://soilwise-he.github.io/soil-health#Ph

In [24]:
# Load the valid property list from the CSV file
def load_property_terms(csv_file_path):
    property_terms = []
    with open(csv_file_path, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            prop_uri = row['uri'].strip()
            label = row['label'].strip()
            property_terms.append({
                'uri': prop_uri,
                'label': label
            })
    return property_terms

# Preprocess the term to add spaces between camel case words
def preprocess_term(term):
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', term)

# Generate embeddings using OpenAI API
def get_embedding(text):
    preprocessed_text = preprocess_term(text)
    response = client.embeddings.create(
    model="text-embedding-3-large",
    input=preprocessed_text
    )
    return np.array(response.data[0].embedding)

# Precompute embeddings for valid property terms using only label
def embed_valid_terms(valid_terms):
    embeddings = {}
    for term in valid_terms:
        text_to_embed = term['label']
        embeddings[term['uri']] = get_embedding(text_to_embed)
    return embeddings

# Extract the term from the URI, handling both '#' and '/'
def extract_term_from_uri(uri):
    return uri.split("#")[-1] if "#" in uri else uri.split("/")[-1]

# Compare custom terms to valid terms using cosine similarity with precomputed embeddings
def find_closest_term(custom_term, valid_terms, valid_embeddings):
    custom_embedding = get_embedding(custom_term)
    valid_term_list = [term['uri'] for term in valid_terms]
    valid_embeddings_list = [valid_embeddings[term] for term in valid_term_list]
    
    similarities = cosine_similarity([custom_embedding], valid_embeddings_list)[0]
    best_match_index = np.argmax(similarities)
    
    return valid_term_list[best_match_index], similarities[best_match_index]

# Function to check if predicates are in the property list before embedding
def check_predicates_in_property_list(predicates, valid_uris):
    predicates_not_in_list = [pred for pred in predicates if pred not in valid_uris]
    return predicates_not_in_list

# Main function to check and correct the RDF knowledge graph
def check_and_correct_rdf(graph, csv_file_path):
    # Step 1: Load valid property terms
    property_terms = load_property_terms(csv_file_path)
    valid_uris = [term['uri'] for term in property_terms]

    # Step 2: Extract all predicates from the graph
    predicates = set(str(pred) for _, pred, _ in graph)

    # Step 3: Check if there are predicates not in the property list
    predicates_not_in_list = check_predicates_in_property_list(predicates, valid_uris)

    # Step 4: Precompute embeddings for valid property terms if needed
    if predicates_not_in_list:
        valid_term_embeddings = embed_valid_terms(property_terms)
    else:
        valid_term_embeddings = {}
    
    # Step 5: Iterate over all predicates and check if they are valid
    for s, pred, o in graph:
        pred_str = str(pred)
        if pred_str in predicates_not_in_list:
            # Extract the term from the URI (handling both '#' and '/')
            predicate_short = extract_term_from_uri(pred_str)
            # Find the closest valid term
            closest_term, similarity = find_closest_term(predicate_short, property_terms, valid_term_embeddings)
            print(predicate_short, extract_term_from_uri(closest_term), similarity)
            if similarity > 0.5:  # Threshold for correction
                # Replace the wrong predicate with the closest valid term
                graph.add((s, rdflib.URIRef(closest_term), o))
                graph.remove((s, pred, o))
    
    return graph

g_0 = check_and_correct_rdf(g_0, 'in_ontovocabs/properties.csv')
print(g_0.serialize(format="turtle"))

hasCharacteristic RO_0000053 0.8989138992564535
involves RO_0002331 0.7534612234365106
hasClimate hasScale 0.3354317150232815
hasCharacteristic RO_0000053 0.8989138992564535
formIn forms 0.6807255182999736
foundAlong atLocation 0.47495647545831876
hasCharacteristic RO_0000053 0.8989309455289812
includes includes 0.9999999999999998
occursIn BFO_0000066 0.8647597749210187
influencedBy isAffectedBy 0.5940053050241392
foundAlong atLocation 0.47495647545831876
hasSoils hasBiomass 0.34675528423309765
developPrimarilyIn isDevelopedFor 0.5467938205617837
hasClimate hasScale 0.3354317150232815
hasComponent hasComponent 0.9200868720347719
influencedBy isAffectedBy 0.5940053050241392
suitableFor isAvailableFor 0.5266149925537095
_3 object 0.36890549323489447
hasTypicalSoilType hasType 0.40442922680130344
suitableFor isAvailableFor 0.5266149925537095
hasClimate hasScale 0.3354317150232815
hasRegion AFX_0002696 0.484933809685344
proneTo RO_0000052 0.3970700009667062
hasCharacteristic RO_0000053 0.8

### Post-process the knowledge graph

In [ ]:
def parse_turtle(turtle_string):
    g = Graph()
    g.parse(data=turtle_string, format='turtle')
    return g

def merge_graphs(json_data):
    merged_graph = Graph()
    
    for item in json_data['dataset']:
        turtle_string = item['rdf_graph_turtle']
        graph = parse_turtle(turtle_string)
        merged_graph += graph
    
    return merged_graph

def remove_duplicates(graph):
    # Create a new graph to store unique triples
    unique_graph = Graph()
    
    # Iterate through all triples in the original graph
    for s, p, o in graph:
        # Check if the triple already exists in the unique graph
        if not (s, p, o) in unique_graph:
            unique_graph.add((s, p, o))
    
    return unique_graph

def main(file_path):
    merged_graph = merge_graphs(data)
    unique_graph = remove_duplicates(merged_graph)

    unique_graph.bind("agrontology", agrontology)
    unique_graph.bind("qudt", qudt)
    unique_graph.bind("she", she)
    unique_graph.bind("unit", unit)
    unique_graph.bind("sio", sio)
    unique_graph.bind("obo", obo)
    unique_graph.bind("wdt", wdt)
    unique_graph.bind("biolink", biolink)
    unique_graph.bind("af-x", afox)
    unique_graph.bind("af-r", afor)
    unique_graph.bind("sorelpr", sorelpr)
    unique_graph.bind("sorelsc", sorelsc)
    unique_graph.bind("sohuj", sohuj)
    unique_graph.bind("sorelph", sorelph)
    unique_graph.bind("sorelm", sorelm)
    unique_graph.bind("bao", bao)
    unique_graph.bind("repr", repr)
    unique_graph.bind("sorepsg", sorepsg)
    unique_graph.bind("sorelch", sorelch)
    unique_graph.bind("sorelsp", sorelsp)
    unique_graph.bind("om", om)

    return unique_graph

if __name__ == "__main__":
    unique_graph = main("text_RDF_gs.json")

In [17]:
def add_inverse_triples(graph, csv_file_path):
    # Load invertible properties from CSV file
    invertible_properties = {}
    with open(csv_file_path, 'r') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for row in reader:
            if len(row) == 2:
                prop, inverse = row
                prop = prop.strip()
                inverse = inverse.strip()
                invertible_properties[prop] = inverse
                invertible_properties[inverse] = prop  # Add bidirectional mapping

    # Add triples with inverted properties
    new_triples = []

    for s, p, o in graph:
        property_uri = str(p)
        # Check if this property is in our invertible set
        if property_uri in invertible_properties:
            # Skip triples where subject or object is a BNode, or object is a literal
            if isinstance(s, rdflib.BNode) or isinstance(o, rdflib.BNode) or isinstance(o, rdflib.Literal):
                continue

            # Otherwise, add inverse triple
            inverse_property_uri = invertible_properties[property_uri]
            inverse_property = rdflib.URIRef(inverse_property_uri)
            new_triples.append((o, inverse_property, s))

    # Add the new triples to the graph
    for subj, pred, obj in new_triples:
        graph.add((subj, pred, obj))

    return graph

unique_graph = add_inverse_triples(unique_graph, 'in_ontovocabs/prop_mapping.csv')

In [22]:
# Function to normalize British English to American English
def normalize_uk_to_us(label):
    for uk_spelling, us_spelling in uk_us:
        # Use regular expressions to replace only full words
        label = re.sub(rf'\b{uk_spelling}\b', us_spelling, label)
    return label

def concept2vocabs(g, file_path):
    # Load the CSV file
    pref_label_map = {}
    alt_label_map = {}
    with open(file_path, encoding="utf8") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            new_uri = row['concept']
            pref_label = row['prefLabel'].lower()
            alt_labels = row['altLabels'].split(';') if row['altLabels'] else []
            pref_label_map[pref_label] = new_uri
            for alt_label in alt_labels:
                alt_label = alt_label.strip().lower()  # Clean and normalize altLabel
                if alt_label:
                    alt_label_map[alt_label] = new_uri

    # Iterate over the graph and process matching prefLabel and altLabel
    for s, p, o in g:
        label = str(o).lower()
        normalized_label = normalize_uk_to_us(label)  # Apply normalization

        if p == SKOS.prefLabel and normalized_label in pref_label_map:
            # Add skos:exactMatch triples
            exact_match_uri = rdflib.URIRef(pref_label_map[normalized_label])
            g.add((s, SKOS.exactMatch, exact_match_uri))
        elif p == SKOS.prefLabel and normalized_label in alt_label_map:
            # Add skos:closeMatch triples
            close_match_uri = rdflib.URIRef(alt_label_map[normalized_label])
            g.add((s, SKOS.closeMatch, close_match_uri))

    return g

In [30]:
# Link to AGROVOC vocabs
unique_graph = concept2vocabs(unique_graph, "ex_ontovocabs/agrovoc.csv")
unique_graph.bind("agrovoc", agrovoc)

In [31]:
# Link to ISO11074
unique_graph = concept2vocabs(unique_graph, "ex_ontovocabs/ISO11074v2025.csv")
unique_graph.bind("iso11074", iso11074)

In [32]:
# Link to INRAE Thesaurus
unique_graph = concept2vocabs(unique_graph, "ex_ontovocabs/inrae.csv")
unique_graph.bind("inrae", inrae)

In [33]:
# Link to GEMET Thesaurus
unique_graph = concept2vocabs(unique_graph, "ex_ontovocabs/gemet.csv")
unique_graph.bind("gemet", gemet)

In [128]:
for subj, pred, obj in unique_graph:
    if isinstance(obj, rdflib.Literal):
        # Check if the literal is a pure number or already has a datatype
        if obj.datatype is None and not re.match(r'^\d+$', str(obj)):
            # Add language tag to string literals (except pure numbers and typed literals)
            new_obj = rdflib.Literal(obj, lang='en')
            unique_graph.add((subj, pred, new_obj))
            unique_graph.remove((subj, pred, obj))

In [20]:
# Serialize the final graph to Turtle format
unique_graph.serialize(destination="soil_health_KG.ttl", format="turtle")

<Graph identifier=Necfb9fe90bc048118988b2397f0f6403 (<class 'rdflib.graph.Graph'>)>

In [19]:
def get_match_statistics(ttl_file_path):
    """
    Get statistics on all entities with exactMatch or closeMatch properties.
    
    Args:
        ttl_file_path (str): Path to the TTL file containing the RDF knowledge graph
    """
    
    # Load the graph
    g = Graph()
    g.parse(ttl_file_path, format='turtle')
    
    # Collect unique entities with matches
    entities_with_matches = set()
    
    # Add entities with exactMatch
    for entity in g.subjects(SKOS.exactMatch, None):
        if isinstance(entity, URIRef):
            entities_with_matches.add(entity)
    
    # Add entities with closeMatch
    for entity in g.subjects(SKOS.closeMatch, None):
        if isinstance(entity, URIRef):
            entities_with_matches.add(entity)
    
    # Print statistics
    print(f"Number of unique entities with matches: {len(entities_with_matches)}")

# Example usage
if __name__ == "__main__":
    ttl_file = "soil_health_KG.ttl"  # Replace with your TTL file path
    get_match_statistics(ttl_file)

Number of unique entities with matches: 493


#### Create SKOS version of SHKG

In [19]:
# Add skos:inScheme triple for every skos:Concept in the unique_graph
soil_health_scheme = URIRef("https://soilwise-he.github.io/soil-health")

# Find all skos:Concepts in the graph
concepts = unique_graph.subjects(RDF.type, SKOS.Concept)

# Count how many triples we'll add
concept_count = 0
for concept in concepts:
    concept_count += 1

print(f"Found {concept_count} skos:Concepts in the graph")

# Add the skos:inScheme triple for each concept
triples_added = 0
for concept in unique_graph.subjects(RDF.type, SKOS.Concept):
    # Add the triple: concept skos:inScheme <https://soilwise-he.github.io/soil-health>
    unique_graph.add((concept, SKOS.inScheme, soil_health_scheme))
    triples_added += 1

print(f"Added {triples_added} skos:inScheme triples to the graph")

Found 1786 skos:Concepts in the graph
Added 1786 skos:inScheme triples to the graph


In [24]:
# Find minimum set of top concepts to ensure all concepts are accessible through hierarchy
# This uses a graph algorithm to find the minimum dominating set

def find_minimal_top_concepts(graph):
    """
    Find the minimal set of concepts that need to be marked as top concepts
    to ensure all concepts in the graph are accessible through skos:narrower relationships.
    """
    # Get all concepts
    all_concepts = set(graph.subjects(RDF.type, SKOS.Concept))
    print(f"Total concepts in graph: {len(all_concepts)}")
    
    # Build a mapping of broader->narrower relationships
    narrower_map = {}  # concept -> set of concepts that are narrower
    broader_map = {}   # concept -> set of concepts that are broader
    
    for concept in all_concepts:
        narrower_map[concept] = set()
        broader_map[concept] = set()
    
    # Populate the maps using skos:narrower and skos:broader relationships
    for s, p, o in graph:
        if p == SKOS.narrower and s in all_concepts and o in all_concepts:
            narrower_map[s].add(o)
            broader_map[o].add(s)
        elif p == SKOS.broader and s in all_concepts and o in all_concepts:
            broader_map[s].add(o)
            narrower_map[o].add(s)
    
    # Find concepts that have no broader concepts (potential roots)
    root_candidates = {concept for concept in all_concepts if len(broader_map[concept]) == 0}
    print(f"Concepts with no broader concepts: {len(root_candidates)}")
    
    # Function to get all reachable concepts from a given concept via narrower relationships
    def get_reachable_concepts(start_concept, visited=None):
        if visited is None:
            visited = set()
        
        if start_concept in visited:
            return visited
        
        visited.add(start_concept)
        
        # Recursively visit all narrower concepts
        for narrower_concept in narrower_map[start_concept]:
            get_reachable_concepts(narrower_concept, visited)
        
        return visited
    
    # Find the minimal set of top concepts using a greedy approach
    uncovered_concepts = all_concepts.copy()
    top_concepts = []
    
    while uncovered_concepts:
        # Find the concept that covers the most uncovered concepts
        best_concept = None
        best_coverage = 0
        
        for candidate in root_candidates:
            if candidate not in uncovered_concepts:
                continue
                
            reachable = get_reachable_concepts(candidate)
            coverage = len(reachable & uncovered_concepts)
            
            if coverage > best_coverage:
                best_coverage = coverage
                best_concept = candidate
        
        # If no root candidate covers uncovered concepts, pick any uncovered concept
        if best_concept is None:
            # Find any uncovered concept and make it a top concept
            best_concept = next(iter(uncovered_concepts))
            reachable = get_reachable_concepts(best_concept)
        else:
            reachable = get_reachable_concepts(best_concept)
        
        top_concepts.append(best_concept)
        uncovered_concepts -= reachable
        
        print(f"Selected top concept: {best_concept}")
        print(f"  Covers {len(reachable)} concepts")
        print(f"  Remaining uncovered: {len(uncovered_concepts)}")
    
    return top_concepts

# Find the minimal top concepts
minimal_top_concepts = find_minimal_top_concepts(unique_graph)
print(f"\nMinimal set of top concepts found: {len(minimal_top_concepts)}")
print("Top concepts:")
for i, concept in enumerate(minimal_top_concepts, 1):
    pref_label = unique_graph.value(concept, SKOS.prefLabel)
    print(f"{i}. {concept} -> {pref_label if pref_label else 'No prefLabel'}")

Total concepts in graph: 1786
Concepts with no broader concepts: 1333
Selected top concept: https://soilwise-he.github.io/soil-health#ObservableSoilProperties
  Covers 36 concepts
  Remaining uncovered: 1750
Selected top concept: https://soilwise-he.github.io/soil-health#SoilIndicators
  Covers 31 concepts
  Remaining uncovered: 1719
Selected top concept: https://soilwise-he.github.io/soil-health#AnalyticalMethodologies
  Covers 33 concepts
  Remaining uncovered: 1688
Selected top concept: https://soilwise-he.github.io/soil-health#BiologicalIndicators
  Covers 23 concepts
  Remaining uncovered: 1667
Selected top concept: https://soilwise-he.github.io/soil-health#UrbanSoils
  Covers 19 concepts
  Remaining uncovered: 1648
Selected top concept: https://soilwise-he.github.io/soil-health#OrganicPollutants
  Covers 14 concepts
  Remaining uncovered: 1634
Selected top concept: https://soilwise-he.github.io/soil-health#EssentialBiodiversityVariables
  Covers 14 concepts
  Remaining uncovered:

In [25]:
# Add the skos:hasTopConcept and skos:topConceptOf relationships to the graph
scheme_uri = URIRef("https://soilwise-he.github.io/soil-health")

print(f"Adding {len(minimal_top_concepts)} top concepts to the scheme...")

# Add the bidirectional relationships for each top concept
for top_concept in minimal_top_concepts:
    # Add: scheme skos:hasTopConcept top_concept
    unique_graph.add((scheme_uri, SKOS.hasTopConcept, top_concept))
    
    # Add: top_concept skos:topConceptOf scheme
    unique_graph.add((top_concept, SKOS.topConceptOf, scheme_uri))
    
    pref_label = unique_graph.value(top_concept, SKOS.prefLabel)
    print(f"Added top concept: {pref_label if pref_label else top_concept}")

print(f"\nSuccessfully added {len(minimal_top_concepts)} top concepts to the scheme")
print("Each top concept now has both:")
print("  - <scheme> skos:hasTopConcept <concept>") 
print("  - <concept> skos:topConceptOf <scheme>")

# Verify the additions
top_concept_count = len(list(unique_graph.subjects(SKOS.topConceptOf, scheme_uri)))
has_top_concept_count = len(list(unique_graph.objects(scheme_uri, SKOS.hasTopConcept)))

print(f"\nVerification:")
print(f"Concepts with skos:topConceptOf: {top_concept_count}")
print(f"Scheme has skos:hasTopConcept: {has_top_concept_count}")
print(f"Match: {top_concept_count == has_top_concept_count == len(minimal_top_concepts)}")

Adding 1333 top concepts to the scheme...
Added top concept: observable soil properties
Added top concept: soil indicators
Added top concept: analytical methodologies
Added top concept: biological indicators
Added top concept: urban soils
Added top concept: organic pollutants
Added top concept: essential biodiversity variables
Added top concept: soil function indicators
Added top concept: soil attributes
Added top concept: soil functions
Added top concept: non-soil indicators
Added top concept: soil types
Added top concept: functional groups
Added top concept: soil threat indicators
Added top concept: commonly regulated heavy metals
Added top concept: derived soil properties
Added top concept: endpoints
Added top concept: complex soil physical parameters
Added top concept: diffuse pollution indicators
Added top concept: soil parameters
Added top concept: soil biodiversity indicators
Added top concept: soil services
Added top concept: land use sub-indicators
Added top concept: human act

In [ ]:
# VALIDATION: Execute SPARQL query to find any orphaned concepts
# This query finds concepts that are NOT reachable from any top concept through broader* paths

validation_query = """
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?concept WHERE {
  ?concept a skos:Concept .
  FILTER NOT EXISTS {
    ?concept skos:broader* ?topConcept .
    ?scheme skos:hasTopConcept ?topConcept .
  }
}
"""

print("Executing validation query to find orphaned concepts...")
print("Query:")
print(validation_query)

# Execute the SPARQL query on the updated graph
results = unique_graph.query(validation_query)

# Convert results to list to count and display
orphaned_concepts = list(results)

print(f"\nValidation Results:")
print(f"Number of orphaned concepts found: {len(orphaned_concepts)}")

if len(orphaned_concepts) == 0:
    print("✅ SUCCESS: No orphaned concepts found!")
    print("✅ All concepts are reachable from at least one top concept.")
    print("✅ OntoPortal will be able to index all concepts in the knowledge graph.")
else:
    print("❌ WARNING: Found orphaned concepts that are not reachable from any top concept:")
    for i, (concept,) in enumerate(orphaned_concepts[:10], 1):  # Show first 10
        pref_label = unique_graph.value(concept, SKOS.prefLabel)
        print(f"  {i}. {concept} -> {pref_label if pref_label else 'No prefLabel'}")
    
    if len(orphaned_concepts) > 10:
        print(f"  ... and {len(orphaned_concepts) - 10} more orphaned concepts")
    
    print("\n❌ These concepts need to be connected to the hierarchy or added as additional top concepts.")

Executing validation query to find orphaned concepts...
Query:

PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?concept WHERE {
  ?concept a skos:Concept .
  FILTER NOT EXISTS {
    ?concept skos:broader* ?topConcept .
    ?scheme skos:hasTopConcept ?topConcept .
  }
}


Validation Results:
Number of orphaned concepts found: 0
✅ SUCCESS: No orphaned concepts found!
✅ All concepts are reachable from at least one top concept.
✅ OntoPortal will be able to index all concepts in the knowledge graph.


In [20]:
unique_graph.serialize(destination="soil_health_SKOS.ttl", format="turtle")

<Graph identifier=N56c846aec101473d8f9649b3c4c01b44 (<class 'rdflib.graph.Graph'>)>

#### Singular labels

In [ ]:
def add_singular_forms_as_altlabels(graph_file, output_file=None):
    # Ensure the required NLTK data is downloaded
    nltk.download('punkt_tab', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    
    # Load the RDF graph
    g = Graph()
    g.parse(graph_file)
    
    # Keep track of new triples to add
    new_triples = []
    
    # Find all skos:prefLabel triples
    for s, pred, o in g.triples((None, SKOS.prefLabel, None)):
        # CONSTRAINT 1: Check if the subject is a skos:Concept
        is_concept = False
        for concept_triple in g.triples((s, RDF.type, SKOS.Concept)):
            is_concept = True
            break
        
        if not is_concept:
            continue  # Skip if not a skos:Concept
        
        # Get the label text
        pref_label = str(o)
        
        # Tokenize and POS tag the label
        tokens = word_tokenize(pref_label)
        tagged_tokens = pos_tag(tokens)
        
        # Process each token
        singular_tokens = []
        for word, tag in tagged_tokens:
            # Check if the word is a plural noun (NNS for plural noun)
            if tag == 'NNS':
                # Get the singular form
                singular_forms = lemminflect.getLemma(word, 'NOUN')
                if singular_forms:
                    singular_tokens.append(singular_forms[0])
                else:
                    singular_tokens.append(word)
            else:
                singular_tokens.append(word)
        
        # Join the tokens back into a phrase
        singular_form = ' '.join(singular_tokens)
        
        # Only add if different from original
        if singular_form != pref_label:
            new_triples.append((s, SKOS.altLabel, Literal(singular_form, lang=o.language if hasattr(o, 'language') else None)))
    
    # Add all new triples to the graph
    for triple in new_triples:
        g.add(triple)
    
    # Save the modified graph
    if output_file:
        g.serialize(destination=output_file, format="turtle")
    
    return g

if __name__ == "__main__":
    graph_file = "soil_health_KG.ttl"
    output_file = "soil_health_KG.ttl"
    
    modified_graph = add_singular_forms_as_altlabels(graph_file, output_file)
    print(f"Successfully added singular forms as altLabels to {output_file}")

Successfully added singular forms as altLabels to soil_health_KG_sing.ttl


#### Ontology in Turtle

In [ ]:
def csv_to_ttl_ontology(classes_file, properties_file, prop_mapping_file, output_file):
    """
    Convert CSV files containing ontology information into a TTL ontology file.
    
    Args:
        classes_file: Path to classes CSV file
        properties_file: Path to properties CSV file  
        prop_mapping_file: Path to property mapping CSV file
        output_file: Path for output TTL file
    """
    
    # Read CSV files
    classes_df = pd.read_csv(classes_file)
    properties_df = pd.read_csv(properties_file)
    prop_mapping_df = pd.read_csv(prop_mapping_file)
    
    # Clean headers (strip whitespace)
    classes_df.columns = classes_df.columns.str.strip()
    properties_df.columns = properties_df.columns.str.strip()
    prop_mapping_df.columns = prop_mapping_df.columns.str.strip()
    
    # Extract namespace prefixes from the data
    prefixes = set()
    
    def extract_prefix(turtle_notation):
        """Extract prefix from turtle notation (e.g., 'skos:Concept' -> 'skos')"""
        if ':' in str(turtle_notation):
            return str(turtle_notation).split(':')[0]
        return None
    
    def get_namespace_from_uri(uri):
        """Extract namespace from full URI"""
        if '#' in uri:
            return uri.split('#')[0] + '#'
        else:
            # For URIs ending with /, take everything up to the last /
            parts = uri.rstrip('/').split('/')
            return '/'.join(parts[:-1]) + '/'
    
    # Collect prefixes and their namespaces
    prefix_to_namespace = {}
    
    # From classes
    for _, row in classes_df.iterrows():
        prefix = extract_prefix(row['class'])
        if prefix:
            namespace = get_namespace_from_uri(row['uri'])
            prefix_to_namespace[prefix] = namespace
    
    # From properties  
    for _, row in properties_df.iterrows():
        prefix = extract_prefix(row['prop'])
        if prefix:
            namespace = get_namespace_from_uri(row['uri'])
            prefix_to_namespace[prefix] = namespace
    
    # Create inverse property mapping for quick lookup
    inverse_map = {}
    for _, row in prop_mapping_df.iterrows():
        prop_uri = str(row['prop']).strip()
        inverse_uri = str(row['inverse']).strip()
        inverse_map[prop_uri] = inverse_uri
        # Add reverse mapping as well
        inverse_map[inverse_uri] = prop_uri
    
    # Start building TTL content
    ttl_content = []
    
    # Add standard prefixes
    ttl_content.extend([
        "@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .",
        "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .",
        "@prefix owl: <http://www.w3.org/2002/07/owl#> .",
        "@prefix skos: <http://www.w3.org/2004/02/skos/core#> .",
    ])
    
    # Add discovered prefixes
    for prefix, namespace in sorted(prefix_to_namespace.items()):
        if prefix not in ['rdf', 'rdfs', 'owl', 'skos']:  # Skip standard prefixes
            ttl_content.append(f"@prefix {prefix}: <{namespace}> .")
    
    ttl_content.append("")  # Empty line after prefixes
    
    # Add classes
    ttl_content.append("# Classes")
    for _, row in classes_df.iterrows():
        class_turtle = str(row['class']).strip()
        label = str(row['label']).strip()
        
        ttl_content.extend([
            f"{class_turtle} a rdfs:Class ;",
            f"    skos:prefLabel \"{label}\" .",
            ""
        ])
    
    # Add properties
    ttl_content.append("# Properties")
    for _, row in properties_df.iterrows():
        prop_turtle = str(row['prop']).strip()
        prop_uri = str(row['uri']).strip()
        label = str(row['label']).strip()
        
        ttl_lines = [
            f"{prop_turtle} a rdf:Property ;",
            f"    skos:prefLabel \"{label}\" ;"
        ]
        
        # Check if this property has an inverse
        if prop_uri in inverse_map:
            inverse_uri = inverse_map[prop_uri]
            # Find the turtle notation for the inverse property
            inverse_turtle = None
            for _, inv_row in properties_df.iterrows():
                if str(inv_row['uri']).strip() == inverse_uri:
                    inverse_turtle = str(inv_row['prop']).strip()
                    break
            
            if inverse_turtle:
                ttl_lines.append(f"    owl:inverseOf {inverse_turtle}")
        
        # Close the property definition
        ttl_lines[-1] += " ."
        ttl_content.extend(ttl_lines)
        ttl_content.append("")
    
    # Write to file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(ttl_content))
    
    print(f"Ontology TTL file created successfully: {output_file}")
    print(f"Classes processed: {len(classes_df)}")
    print(f"Properties processed: {len(properties_df)}")
    print(f"Inverse property pairs: {len(prop_mapping_df)}")

# Usage example
if __name__ == "__main__":
    csv_to_ttl_ontology(
        classes_file="in_ontovocabs/classes.csv",
        properties_file="in_ontovocabs/properties.csv", 
        prop_mapping_file="in_ontovocabs/prop_mapping.csv",
        output_file="in_ontovocabs/ontology.ttl"
    )

Ontology TTL file created successfully: in_ontovocabs/ontology.ttl
Classes processed: 20
Properties processed: 205
Inverse property pairs: 60


#### Entity normalization

In [19]:
def find_plural_singular_concepts(graph_path):
    """
    Parses a SKOS vocabulary in Turtle format and identifies concepts
    that are likely singular/plural forms of each other.

    Args:
        graph_path (str): The file path to the RDF Turtle file.

    Returns:
        list: A list of tuples, where each tuple contains the URIs of
              two concepts that are potential singular/plural matches.
    """
    # Initialize the inflection engine
    p = inflect.engine()

    # Define the SKOS namespace
    SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

    # Create a graph and parse the Turtle file
    g = rdflib.Graph()
    try:
        g.parse(graph_path, format="turtle")
    except FileNotFoundError:
        print(f"Error: The file '{graph_path}' was not found.")
        return []
    except Exception as e:
        print(f"Error parsing the RDF file: {e}")
        return []

    # --- Step 1: Extract all concepts and their preferred labels ---
    # We store them in a dictionary for easy lookup.
    # Key: Concept URI (rdflib.URIRef)
    # Value: Preferred Label (str)
    concept_labels = {}
    for s, o in g.subject_objects(predicate=SKOS.prefLabel):
        # Ensure the subject is a concept and the label is a literal
        if (s, rdflib.RDF.type, SKOS.Concept) in g and isinstance(o, rdflib.Literal):
            concept_labels[s] = str(o)

    print(f"Found {len(concept_labels)} concepts with a skos:prefLabel.")

    # --- Step 2: Group labels to find potential matches ---
    # We'll create a dictionary where the key is a "base" word (singular form)
    # and the value is a list of concept URIs that match that base word.
    singular_to_concepts = defaultdict(list)
    for concept_uri, label in concept_labels.items():
        # Generate the singular form of the label.
        # If the word is already singular, it remains unchanged.
        # For phrases, this attempts to singularize the last word.
        singular_form = p.singular_noun(label)
        if singular_form is False:
            # The library couldn't determine a singular form, so use the original label
            singular_form = label.lower()
        else:
            singular_form = singular_form.lower()
        
        singular_to_concepts[singular_form].append(concept_uri)

    # --- Step 3: Identify groups with more than one concept ---
    # These are our potential singular/plural matches.
    potential_matches = []
    for singular_form, uris in singular_to_concepts.items():
        if len(uris) > 1:
            # We have a group of concepts that map to the same singular form.
            # For simplicity, we'll report all pairs within the group.
            # For a group [A, B, C], this creates pairs (A, B), (A, C), (B, C)
            from itertools import combinations
            for uri1, uri2 in combinations(uris, 2):
                label1 = concept_labels[uri1]
                label2 = concept_labels[uri2]
                print(f"  - Potential Match Found: '{label1}' ({uri1}) and '{label2}' ({uri2})")
                potential_matches.append((uri1, uri2))

    return potential_matches

# --- Main execution block for IDE ---
if __name__ == "__main__":
    # *** IMPORTANT ***
    # Set the path to your knowledge graph file here.
    # For example: "C:/Users/YourName/Documents/my_graph.ttl"
    # Or a relative path if the file is in the same folder: "my_graph.ttl"
    turtle_file = "soil_health_KG.ttl"

    print(f"\nAnalyzing '{turtle_file}' for singular/plural concept pairs...")
    matches = find_plural_singular_concepts(turtle_file)

    if not matches:
        print("\nNo potential singular/plural concept pairs were found.")
    else:
        print(f"\nAnalysis complete. Found {len(matches)} potential pairs.")


Analyzing 'soil_health_KG.ttl' for singular/plural concept pairs...
Found 1779 concepts with a skos:prefLabel.
  - Potential Match Found: 'organic compounds' (https://soilwise-he.github.io/soil-health#SelectedOrganicCompounds) and 'organic compounds' (https://soilwise-he.github.io/soil-health#OrganicCompounds)

Analysis complete. Found 1 potential pairs.


In [ ]:
def get_labels(graph, entity):
    """
    Retrieve all labels (prefLabel and altLabel) for a given entity.
    """
    labels = []
    # Get preferred labels
    for label in graph.objects(entity, SKOS.prefLabel):
        labels.append(str(label))
    # Get alternative labels if any
    for label in graph.objects(entity, SKOS.altLabel):
        labels.append(str(label))
    return labels

def embed_text(text, model="text-embedding-3-large"):
    """
    Get an embedding for the given text using OpenAI's embedding API.
    """
    response = client.embeddings.create(input=[text], model=model)
    embedding = response.data[0].embedding
    return np.array(embedding)

def cosine_similarity(emb1, emb2):
    """
    Compute cosine similarity between two embeddings.
    """
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def main():
    # Load the RDF graph from the Turtle file
    graph = rdflib.Graph()
    ttl_file = "soil_health_KG.ttl"  # update with your filename
    graph.parse(ttl_file, format="turtle")

    # Get all entities that have at least one skos:prefLabel
    entities = set(graph.subjects(SKOS.prefLabel, None))
    entity_data = {}

    # For each entity, embed each label separately.
    for entity in entities:
        labels = get_labels(graph, entity)
        if not labels:
            continue  # skip if no label is found
        embeddings = []
        for label in labels:
            # Uncomment the next line to see progress for each label.
            print(f"Embedding label for {entity}: {label}")
            embedding = embed_text(label)
            embeddings.append(embedding)
        entity_data[entity] = {"labels": labels, "embeddings": embeddings}

    # Calculate the maximum cosine similarity between every pair of entity embeddings.
    similar_pairs = []
    for (ent1, data1), (ent2, data2) in itertools.combinations(entity_data.items(), 2):
        max_sim = -1.0  # initialize to -1 so any similarity replaces it
        for emb1 in data1["embeddings"]:
            for emb2 in data2["embeddings"]:
                sim = cosine_similarity(emb1, emb2)
                if sim > max_sim:
                    max_sim = sim
        similar_pairs.append((max_sim, ent1, ent2))

    # Sort the pairs by similarity in descending order
    similar_pairs.sort(key=lambda x: x[0], reverse=True)

    # Write the similarity results to a CSV file
    csv_filename = "entity_similarity_gpt.csv"
    with open(csv_filename, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["entity1", "entity2", "similarity"])
        for sim, ent1, ent2 in similar_pairs:
            writer.writerow([str(ent1), str(ent2), sim])

    print(f"Similarity results have been stored in {csv_filename}")

if __name__ == "__main__":
    main()

Embedding label for https://soilwise-he.github.io/soil-health#Precipitation: precipitation
Embedding label for https://soilwise-he.github.io/soil-health#SoilMicrobialActivity: soil microbial activity
Embedding label for https://soilwise-he.github.io/soil-health#Trichloroethane: 1,1,1-trichloroethane
Embedding label for https://soilwise-he.github.io/soil-health#HighSoilPStatus: high soil P status
Embedding label for https://soilwise-he.github.io/soil-health#SheetwashErosion: sheetwash erosion
Embedding label for https://soilwise-he.github.io/soil-health#SpeciesDiversity: species diversity
Embedding label for https://soilwise-he.github.io/soil-health#UrbanSprawl: urban sprawl
Embedding label for https://soilwise-he.github.io/soil-health#FoodQualityStandard: food quality standard
Embedding label for https://soilwise-he.github.io/soil-health#IntegratedManagementApproaches: integrated management approaches
Embedding label for https://soilwise-he.github.io/soil-health#CriticalConcentrations:

In [ ]:
def get_labels(graph, entity):
    """
    Retrieve all labels (prefLabel and altLabel) for a given entity.
    """
    labels = []
    # Get preferred labels
    for label in graph.objects(entity, SKOS.prefLabel):
        labels.append(str(label))
    # Get alternative labels if any
    for label in graph.objects(entity, SKOS.altLabel):
        labels.append(str(label))
    return labels

def embed_text(text, model="qwen3-embedding"):
    """
    Get an embedding for the given text using the Ollama qwen3-embedding.
    """

    response = ollama.embed(
        model=model,
        input=text,
    )
    embedding = response.embeddings
    return np.array(embedding)

def cosine_similarity(emb1, emb2):
    """
    Compute cosine similarity between two embeddings.
    """
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def main():
    # Load the RDF graph from the Turtle file
    graph = rdflib.Graph()
    ttl_file = "soil_health_KG.ttl"  # update with your filename
    graph.parse(ttl_file, format="turtle")

    # Get all entities that have at least one skos:prefLabel
    entities = set(graph.subjects(SKOS.prefLabel, None))
    entity_data = {}

    # For each entity, embed each label separately.
    for entity in entities:
        labels = get_labels(graph, entity)
        if not labels:
            continue  # skip if no label is found
        embeddings = []
        for label in labels:
            # Uncomment the next line to see progress for each label.
            print(f"Embedding label for {entity}: {label}")
            embedding = embed_text(label)
            embeddings.append(embedding)
        entity_data[entity] = {"labels": labels, "embeddings": embeddings}

    # Calculate the maximum cosine similarity between every pair of entity embeddings.
    similar_pairs = []
    for (ent1, data1), (ent2, data2) in itertools.combinations(entity_data.items(), 2):
        max_sim = -1.0  # initialize to -1 so any similarity replaces it
        for emb1 in data1["embeddings"]:
            for emb2 in data2["embeddings"]:
                sim = cosine_similarity(emb1, emb2)
                if sim > max_sim:
                    max_sim = sim
        similar_pairs.append((max_sim, ent1, ent2))

    # Sort the pairs by similarity in descending order
    similar_pairs.sort(key=lambda x: x[0], reverse=True)

    # Write the similarity results to a CSV file
    csv_filename = "entity_similarity_qwen.csv"
    with open(csv_filename, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["entity1", "entity2", "similarity"])
        for sim, ent1, ent2 in similar_pairs:
            writer.writerow([str(ent1), str(ent2), sim])

    print(f"Similarity results have been stored in {csv_filename}")

if __name__ == "__main__":
    main()

In [7]:
def cluster_entities(csv_filename, threshold):
    """
    Build an undirected graph from the CSV file and cluster entities
    based on the similarity threshold. An edge is added between two entities
    if their similarity is greater than the threshold.
    """
    G = nx.Graph()
    with open(csv_filename, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            ent1 = row['entity1']
            ent2 = row['entity2']
            similarity = float(row['similarity'])
            if similarity > threshold:
                G.add_edge(ent1, ent2, weight=similarity)
    # Clusters are the connected components of the graph.
    clusters = list(nx.connected_components(G))
    return clusters

def main():
    csv_filename = "entity_similarity_gpt.csv"  # The CSV file from previous code
    threshold = 0.8
    clusters = cluster_entities(csv_filename, threshold)
    
    print("Entity Clusters:")
    for idx, cluster in enumerate(clusters):
        print(f"Cluster {idx+1}: {cluster}")

if __name__ == "__main__":
    main()

Entity Clusters:
Cluster 1: {'https://soilwise-he.github.io/soil-health#OrganicCompounds', 'https://soilwise-he.github.io/soil-health#SelectedOrganicCompounds'}
Cluster 2: {'https://soilwise-he.github.io/soil-health#HeavyMetals', 'https://soilwise-he.github.io/soil-health#MonitoredHeavyMetals'}
Cluster 3: {'https://soilwise-he.github.io/soil-health#Nutrients', 'https://soilwise-he.github.io/soil-health#MonitoredNutrients'}
Cluster 4: {'https://soilwise-he.github.io/soil-health#OnsiteSoilLosses', 'https://soilwise-he.github.io/soil-health#SoilErosionRateAcceleration', 'https://soilwise-he.github.io/soil-health#AnnualSoilLoss', 'https://soilwise-he.github.io/soil-health#SoilErosionRatesPerUnitOfLandCover', 'https://soilwise-he.github.io/soil-health#SoilErosionRates', 'https://soilwise-he.github.io/soil-health#SoilLoss', 'https://soilwise-he.github.io/soil-health#SoilLossRatesByWater', 'https://soilwise-he.github.io/soil-health#OffsiteSoilLosses', 'https://soilwise-he.github.io/soil-healt

In [36]:
def cluster_entities(csv_filename, threshold):
    """
    Build an undirected graph from the CSV file and cluster entities
    based on the similarity threshold. An edge is added between two entities
    if their similarity is greater than the threshold.
    """
    G = nx.Graph()
    with open(csv_filename, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            ent1 = row['entity1']
            ent2 = row['entity2']
            similarity = float(row['similarity'])
            if similarity > threshold:
                G.add_edge(ent1, ent2, weight=similarity)
    # Clusters are the connected components of the graph.
    clusters = list(nx.connected_components(G))
    return clusters

def main():
    csv_filename = "entity_similarity_qwen.csv"  # The CSV file from previous code
    threshold = 0.8
    clusters = cluster_entities(csv_filename, threshold)
    
    print("Entity Clusters:")
    for idx, cluster in enumerate(clusters):
        print(f"Cluster {idx+1}: {cluster}")

if __name__ == "__main__":
    main()

Entity Clusters:
Cluster 1: {'https://soilwise-he.github.io/soil-health#SelectedOrganicCompounds', 'https://soilwise-he.github.io/soil-health#OrganicCompounds'}
Cluster 2: {'https://soilwise-he.github.io/soil-health#LowerSOMLimit', 'https://soilwise-he.github.io/soil-health#UpperSOMLimit'}
Cluster 3: {'https://soilwise-he.github.io/soil-health#SoilLossRates', 'https://soilwise-he.github.io/soil-health#SoilErosionRateAcceleration', 'https://soilwise-he.github.io/soil-health#TolerableSoilLoss', 'https://soilwise-he.github.io/soil-health#SoilErosionRatesPerUnitOfLandCover', 'https://soilwise-he.github.io/soil-health#PotentialSoilErosionRate', 'https://soilwise-he.github.io/soil-health#ActualSoilErosionRate', 'https://soilwise-he.github.io/soil-health#SoilLoss', 'https://soilwise-he.github.io/soil-health#SoilLossInDepth', 'https://soilwise-he.github.io/soil-health#SoilLossRatesByWater', 'https://soilwise-he.github.io/soil-health#SoilErosionRates', 'https://soilwise-he.github.io/soil-health

In [18]:
def check_labels_in_rdf(file_path):
    """
    Check if all entities except references (dcterms:BibliographicResource) have 
    at least one skos:prefLabel, and if any entity has more than one skos:prefLabel.
    
    Args:
        file_path: Path to the Turtle (.ttl) file
    """
    # Initialize graph
    g = Graph()
    
    # Parse the Turtle file
    print(f"Parsing {file_path}...")
    g.parse(file_path, format="turtle")
    print(f"Parsed {len(g)} triples.")
    
    # Define relevant namespaces and predicates
    skos_prefLabel = SKOS.prefLabel
    dcterms_BibliographicResource = DCTERMS.BibliographicResource
    
    # Get all entities (subjects with an RDF.type predicate)
    all_entities = set()
    for s, p, o in g.triples((None, RDF.type, None)):
        # We're only interested in URIs and blank nodes as entities, not literals
        if isinstance(s, (URIRef, BNode)):
            all_entities.add(s)
    
    # Identify references (entities of type dcterms:BibliographicResource)
    references = set()
    for s, p, o in g.triples((None, RDF.type, dcterms_BibliographicResource)):
        references.add(s)
    
    # Non-reference entities
    non_references = all_entities - references
    
    # Check if all non-references have at least one skos:prefLabel
    entities_without_label = set()
    for entity in non_references:
        labels = list(g.objects(entity, skos_prefLabel))
        if not labels:
            entities_without_label.add(entity)
    
    # Check if any entity has more than one skos:prefLabel
    entities_with_multiple_labels = []
    for entity in all_entities:
        labels = list(g.objects(entity, skos_prefLabel))
        if len(labels) > 1:
            entities_with_multiple_labels.append((entity, labels))
    
    # Print summary
    print("\n=== Summary ===")
    print(f"Total entities: {len(all_entities)}")
    print(f"References (dcterms:BibliographicResource): {len(references)}")
    print(f"Non-references: {len(non_references)}")
    print(f"Non-reference entities without skos:prefLabel: {len(entities_without_label)}")
    print(f"Entities with multiple skos:prefLabel values: {len(entities_with_multiple_labels)}")
    
    # Print details about entities without labels
    print("\n=== Entities without skos:prefLabel ===")
    if entities_without_label:
        for i, entity in enumerate(entities_without_label, 1):
            # Get the types of this entity
            types = [str(o) for o in g.objects(entity, RDF.type)]
            print(f"  {i}. {entity}")
            print(f"     Types: {', '.join(types)}")
    else:
        print("All non-reference entities have at least one skos:prefLabel.")
    
    # Print details about entities with multiple labels
    print("\n=== Entities with multiple skos:prefLabel values ===")
    if entities_with_multiple_labels:
        for i, (entity, labels) in enumerate(entities_with_multiple_labels, 1):
            # Get the types of this entity
            types = [str(o) for o in g.objects(entity, RDF.type)]
            is_reference = entity in references
            
            print(f"  {i}. {entity}")
            print(f"     Types: {', '.join(types)}")
            print(f"     Is reference: {is_reference}")
            print(f"     Labels ({len(labels)}):")
            for j, label in enumerate(labels, 1):
                if isinstance(label, Literal):
                    lang = f"@{label.language}" if label.language else ""
                    print(f"       {j}. \"{label}\"{lang}")
                else:
                    print(f"       {j}. {label}")
    else:
        print("No entities with multiple skos:prefLabel values found.")

# Set your file path here
file_path = "soil_health_KG.ttl"  # CHANGE THIS TO YOUR FILE PATH

# Run the script
if __name__ == "__main__":
    check_labels_in_rdf(file_path)

Parsing soil_health_KG.ttl...
Parsed 11719 triples.

=== Summary ===
Total entities: 2017
References (dcterms:BibliographicResource): 158
Non-references: 1859
Non-reference entities without skos:prefLabel: 0
Entities with multiple skos:prefLabel values: 0

=== Entities without skos:prefLabel ===
All non-reference entities have at least one skos:prefLabel.

=== Entities with multiple skos:prefLabel values ===
No entities with multiple skos:prefLabel values found.
Parsed 11719 triples.

=== Summary ===
Total entities: 2017
References (dcterms:BibliographicResource): 158
Non-references: 1859
Non-reference entities without skos:prefLabel: 0
Entities with multiple skos:prefLabel values: 0

=== Entities without skos:prefLabel ===
All non-reference entities have at least one skos:prefLabel.

=== Entities with multiple skos:prefLabel values ===
No entities with multiple skos:prefLabel values found.


#### Relation disambiguation

In [102]:
def load_rdf_file(file_path):
    """Load an RDF file into a graph."""
    g = Graph()
    
    try:
        g.parse(file_path)
        print(f"Successfully loaded {file_path}")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
    
    return g

def extract_skos_hierarchy(g):
    """Extract SKOS hierarchical relationships into a directed graph.
    
    In the resulting graph:
    - An edge from A to B means A is broader than B (or B is narrower than A)
    """
    # Create a directed graph where edge A->B means A is broader than B
    G = nx.DiGraph()
    
    # Extract all skos:broader relationships
    # If A skos:broader B, then B is broader than A (edge B->A)
    for s, p, o in g.triples((None, SKOS.broader, None)):
        G.add_edge(str(o), str(s))  # o is broader than s
    
    # Extract all skos:narrower relationships
    # If A skos:narrower B, then A is broader than B (edge A->B)
    for s, p, o in g.triples((None, SKOS.narrower, None)):
        G.add_edge(str(s), str(o))  # s is broader than o
    
    return G

def detect_transitive_inconsistencies(G):
    """Detect inconsistencies where a concept has multiple paths to another concept."""
    inconsistencies = []
    
    # For each node in the graph
    for node in G.nodes():
        # Get direct children
        direct_children = list(G.successors(node))
        
        # For each direct child
        for child in direct_children:
            # Check if any other direct child is an ancestor of this child
            for other_child in direct_children:
                if other_child != child and nx.has_path(G, other_child, child):
                    # Found inconsistency: node -> child and node -> other_child -> ... -> child
                    inconsistencies.append({
                        'type': 'transitive_inconsistency',
                        'node': node,
                        'direct_child': child,
                        'mediating_child': other_child,
                        'description': f"{node} has direct relationship to {child}, "
                                      f"but this should be mediated through {other_child}"
                    })
    
    return inconsistencies

def detect_cycle_inconsistencies(G):
    """Detect cycles in the hierarchy, which represent contradictory relationships."""
    cycle_inconsistencies = []
    
    try:
        cycles = list(nx.simple_cycles(G))
        for cycle in cycles:
            cycle_inconsistencies.append({
                'type': 'cycle_inconsistency',
                'nodes': cycle,
                'description': f"Cycle detected: {' -> '.join(cycle)} -> {cycle[0]}"
            })
    except nx.NetworkXNoCycle:
        # No cycles found
        pass
    
    return cycle_inconsistencies

def detect_multiple_broader_concepts(G):
    """Detect cases where a concept has multiple broader concepts 
    that aren't in a hierarchical relationship with each other."""
    inconsistencies = []
    
    # For each node in the graph
    for node in G.nodes():
        # Get direct broader concepts (predecessors)
        broader_concepts = list(G.predecessors(node))
        
        # If there are multiple broader concepts
        if len(broader_concepts) > 1:
            unrelated_pairs = []
            
            # Check each pair of broader concepts
            for i in range(len(broader_concepts)):
                for j in range(i+1, len(broader_concepts)):
                    concept1 = broader_concepts[i]
                    concept2 = broader_concepts[j]
                    
                    # Check if there's no hierarchical relationship between them
                    if (not nx.has_path(G, concept1, concept2) and 
                        not nx.has_path(G, concept2, concept1)):
                        unrelated_pairs.append((concept1, concept2))
            
            if unrelated_pairs:
                inconsistencies.append({
                    'type': 'multiple_broader_concepts',
                    'node': node,
                    'broader_concepts': broader_concepts,
                    'unrelated_pairs': unrelated_pairs,
                    'description': f"{node} has multiple unrelated broader concepts: {', '.join(broader_concepts)}"
                })
    
    return inconsistencies

def suggest_corrections(inconsistencies):
    """Suggest corrections for the detected inconsistencies."""
    corrections = []
    
    for inconsistency in inconsistencies:
        if inconsistency['type'] == 'transitive_inconsistency':
            node = inconsistency['node']
            direct_child = inconsistency['direct_child']
            mediating_child = inconsistency['mediating_child']
            
            # Find the shortest path through the mediating child
            correction = {
                'type': 'remove_direct_relationship',
                'from': node,
                'to': direct_child,
                'description': f"Remove direct relationship: {node} skos:narrower {direct_child}. "
                               f"The relationship is already established through: "
                               f"{node} skos:narrower {mediating_child} ... skos:narrower {direct_child}"
            }
            corrections.append(correction)
        
        elif inconsistency['type'] == 'cycle_inconsistency':
            nodes = inconsistency['nodes']
            
            correction = {
                'type': 'resolve_cycle',
                'nodes': nodes,
                'description': f"Resolve cycle by removing one of the relationships in: {' -> '.join(nodes)} -> {nodes[0]}. "
                               f"Review and determine the correct hierarchical structure."
            }
            corrections.append(correction)
        
        elif inconsistency['type'] == 'multiple_broader_concepts':
            node = inconsistency['node']
            broader_concepts = inconsistency['broader_concepts']
            unrelated_pairs = inconsistency['unrelated_pairs']
            
            correction = {
                'type': 'review_multiple_parents',
                'node': node,
                'broader_concepts': broader_concepts,
                'description': f"Review the multiple broader concepts for {node}: {', '.join(broader_concepts)}. "
                               f"Determine if these should be related in a hierarchy or if one relationship should be removed."
            }
            corrections.append(correction)
    
    return corrections

def format_results(results):
    """Format the results for display."""
    output = []
    
    # Count inconsistencies
    total_inconsistencies = len(results['inconsistencies']['all'])
    output.append(f"SKOS Relationship Inconsistency Report")
    output.append(f"======================================")
    output.append(f"\nFound {total_inconsistencies} inconsistencies in total:")
    output.append(f"- {len(results['inconsistencies']['transitive'])} transitive inconsistencies")
    output.append(f"- {len(results['inconsistencies']['cycles'])} cycle inconsistencies")
    output.append(f"- {len(results['inconsistencies']['multiple_broader'])} multiple broader concept inconsistencies")
    
    # Print each inconsistency with its suggested correction
    for i, inconsistency in enumerate(results['inconsistencies']['all']):
        output.append(f"\n--- Inconsistency {i+1} ---")
        output.append(f"Type: {inconsistency['type']}")
        output.append(f"Description: {inconsistency['description']}")
        
        # Find matching correction
        for correction in results['corrections']:
            if (('node' in inconsistency and 'from' in correction and inconsistency['node'] == correction['from']) or
                ('nodes' in inconsistency and 'nodes' in correction and inconsistency['nodes'] == correction['nodes'])):
                output.append(f"Suggested correction: {correction['description']}")
                break
    
    return "\n".join(output)

def main(file_path):
    """Main function to run the inconsistency detection and correction suggestion."""
    # Step 1: Load RDF file
    g = load_rdf_file(file_path)
    
    # Step 2: Extract SKOS hierarchy into a directed graph
    G = extract_skos_hierarchy(g)
    
    # Count nodes and edges
    print(f"Extracted graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    
    # Step 3: Detect inconsistencies
    transitive_inconsistencies = detect_transitive_inconsistencies(G)
    cycle_inconsistencies = detect_cycle_inconsistencies(G)
    multiple_broader_inconsistencies = detect_multiple_broader_concepts(G)
    
    all_inconsistencies = (transitive_inconsistencies + 
                          cycle_inconsistencies + 
                          multiple_broader_inconsistencies)
    
    # Step 4: Suggest corrections
    corrections = suggest_corrections(all_inconsistencies)
    
    # Print results
    results = {
        'inconsistencies': {
            'transitive': transitive_inconsistencies,
            'cycles': cycle_inconsistencies,
            'multiple_broader': multiple_broader_inconsistencies,
            'all': all_inconsistencies
        },
        'corrections': corrections
    }
    
    print(format_results(results))
    
    return {
        'graph': G,
        'inconsistencies': results['inconsistencies'],
        'corrections': corrections
    }

if __name__ == "__main__":
    # Example usage with a single merged knowledge graph file
    file_path = "soil_health_KG.ttl"  # Replace with your actual file path
    
    results = main(file_path)

Successfully loaded soil_health_KG.ttl
Extracted graph has 596 nodes and 593 edges
SKOS Relationship Inconsistency Report

Found 78 inconsistencies in total:
- 11 transitive inconsistencies
- 0 cycle inconsistencies
- 67 multiple broader concept inconsistencies

--- Inconsistency 1 ---
Type: transitive_inconsistency
Description: https://soilwise-he.github.io/soil-health#ObservableSoilProperties has direct relationship to https://soilwise-he.github.io/soil-health#SoilNitrogen, but this should be mediated through https://soilwise-he.github.io/soil-health#SoilAnionContent
Suggested correction: Remove direct relationship: https://soilwise-he.github.io/soil-health#ObservableSoilProperties skos:narrower https://soilwise-he.github.io/soil-health#SoilNitrogen. The relationship is already established through: https://soilwise-he.github.io/soil-health#ObservableSoilProperties skos:narrower https://soilwise-he.github.io/soil-health#SoilAnionContent ... skos:narrower https://soilwise-he.github.io/

In [39]:
def extract_skos_subgraph(rdf_graph):
    """Extract a subgraph containing only SKOS relationships."""
    skos_subgraph = Graph()
    for s, p, o in rdf_graph:
        if p in [SKOS.narrower, SKOS.broader, SKOS.related]:
            skos_subgraph.add((s, p, o))
    return skos_subgraph

def build_hierarchy_graph(skos_subgraph):
    """
    Build a directed graph representing hierarchical relationships.
    Direction: if A -> B, then A is broader than B (B is narrower than A).
    """
    hierarchy_graph = nx.DiGraph()
    
    # Add all concepts as nodes first to ensure isolated nodes are included
    for s, _, o in skos_subgraph:
        hierarchy_graph.add_node(s)
        hierarchy_graph.add_node(o)
    
    # Add edges for hierarchical relationships
    for s, p, o in skos_subgraph:
        if p == SKOS.narrower:
            hierarchy_graph.add_edge(s, o)  # s is broader than o
        elif p == SKOS.broader:
            hierarchy_graph.add_edge(o, s)  # o is broader than s
    
    return hierarchy_graph

def collect_related_pairs(skos_subgraph):
    """Collect all pairs of concepts that are marked as related."""
    related_pairs = set()
    for s, p, o in skos_subgraph:
        if p == SKOS.related:
            related_pairs.add((s, o))
    return related_pairs

def find_hierarchical_path(hierarchy_graph, concept_a, concept_b):
    """Find a path in the hierarchy from concept_a to concept_b if one exists."""
    try:
        if nx.has_path(hierarchy_graph, concept_a, concept_b):
            return nx.shortest_path(hierarchy_graph, concept_a, concept_b)
    except nx.NetworkXError:
        # Handle case where nodes aren't in the graph
        pass
    return None

def collect_path_triples(skos_subgraph, path):
    """Collect the SKOS triples that form a given path."""
    path_triples = []
    for i in range(len(path) - 1):
        source, target = path[i], path[i+1]
        for s, p, o in skos_subgraph:
            if (p == SKOS.narrower and s == source and o == target) or \
               (p == SKOS.broader and s == target and o == source):
                path_triples.append((s, p, o))
                break
    return path_triples

def detect_skos_inconsistencies(rdf_file_path, format=None):
    """
    Detect inconsistencies in SKOS relationships in an RDF knowledge graph.
    
    Args:
        rdf_file_path: Path to the RDF file containing the integrated knowledge graph.
        format: Format of the RDF file (e.g., 'turtle', 'xml', 'n3'). If None, rdflib will try to guess.
    
    Returns:
        A list of inconsistency reports.
    """
    # Load the RDF graph
    rdf_graph = Graph()
    rdf_graph.parse(rdf_file_path, format=format)
    
    # Extract the SKOS subgraph
    skos_subgraph = extract_skos_subgraph(rdf_graph)
    
    # Build the hierarchy graph
    hierarchy_graph = build_hierarchy_graph(skos_subgraph)
    
    # Collect all related pairs
    related_pairs = collect_related_pairs(skos_subgraph)
    
    # Find inconsistencies
    inconsistencies = []
    
    for a, c in related_pairs:
        if a == c:  # Skip self-relations
            continue
        
        # Check for path from a to c (a is broader than c)
        path_a_to_c = find_hierarchical_path(hierarchy_graph, a, c)
        if path_a_to_c:
            path_triples = collect_path_triples(skos_subgraph, path_a_to_c)
            inconsistencies.append({
                "related_pair": (a, c),
                "path": path_a_to_c,
                "direction": "broader",
                "path_triples": path_triples
            })
            continue
        
        # Check for path from c to a (c is broader than a)
        path_c_to_a = find_hierarchical_path(hierarchy_graph, c, a)
        if path_c_to_a:
            path_triples = collect_path_triples(skos_subgraph, path_c_to_a)
            inconsistencies.append({
                "related_pair": (a, c),
                "path": path_c_to_a,
                "direction": "narrower",
                "path_triples": path_triples
            })
    
    # Format the inconsistencies for readability
    formatted_reports = []
    
    for inconsistency in inconsistencies:
        a, c = inconsistency["related_pair"]
        path = inconsistency["path"]
        direction = inconsistency["direction"]
        path_triples = inconsistency["path_triples"]
        
        path_str = " -> ".join(str(node) for node in path)
        triples_str = "\n  ".join(f"{s} {p} {o}" for s, p, o in path_triples)
        
        if direction == "broader":
            formatted_report = (
                f"Inconsistency: {a} and {c} are marked as 'related', but {a} is broader than {c} via path:\n"
                f"  {path_str}\n"
                f"Based on triples:\n  {triples_str}"
            )
        else:  # direction == "narrower"
            formatted_report = (
                f"Inconsistency: {a} and {c} are marked as 'related', but {a} is narrower than {c} via path:\n"
                f"  {path_str}\n"
                f"Based on triples:\n  {triples_str}"
            )
        
        formatted_reports.append(formatted_report)
    
    return formatted_reports

def print_inconsistency_report(inconsistencies, output_file=None):
    """
    Print inconsistencies to console or save to a file.
    
    Args:
        inconsistencies: List of inconsistency reports
        output_file: Optional file path to save the report
    """
    if not inconsistencies:
        print("No inconsistencies found.")
        if output_file:
            with open(output_file, 'w') as f:
                f.write("No inconsistencies found.\n")
        return
    
    print(f"Found {len(inconsistencies)} inconsistencies:")
    
    if output_file:
        with open(output_file, 'w') as f:
            f.write(f"Found {len(inconsistencies)} inconsistencies:\n")
            for i, report in enumerate(inconsistencies, 1):
                f.write(f"\nInconsistency {i}:\n")
                f.write(report + "\n")
        print(f"Report saved to {output_file}")
    else:
        for i, report in enumerate(inconsistencies, 1):
            print(f"\nInconsistency {i}:")
            print(report)

# ----------------------------------------------------------------------
# CONFIGURATION SECTION - MODIFY THESE VARIABLES TO MATCH YOUR SETUP
# ----------------------------------------------------------------------

# Path to your RDF file
INPUT_FILE = "soil_health_KG.ttl"

# Format of the RDF file (e.g., 'turtle', 'xml', 'n3') - set to None to auto-detect
FILE_FORMAT = None

# Path to save the inconsistency report (set to None to print to console only)
OUTPUT_FILE = None  # or "inconsistencies.txt" to save to a file

# ----------------------------------------------------------------------
# EXAMPLE CODE: To analyze a specific RDF file
# ----------------------------------------------------------------------

# Example with a string literal RDF content for testing purposes
def create_test_graph():
    """Create a test graph with inconsistent SKOS relationships"""
    
    # Create a new graph
    g = Graph()
    
    # Define namespaces
    SHE = Namespace("https://soilwise-he.github.io/soil-health#")
    
    # Add triples with inconsistent relationships
    g.add((SHE.ConceptA, SKOS.narrower, SHE.ConceptB))
    g.add((SHE.ConceptA, SKOS.related, SHE.ConceptC))
    g.add((SHE.ConceptB, SKOS.narrower, SHE.ConceptC))
    
    # Save test graph to a file
    test_file = "test_skos_graph.ttl"
    g.serialize(destination=test_file, format="turtle")
    
    return test_file

# ----------------------------------------------------------------------
# EXECUTION SECTION - RUN THIS SCRIPT IN YOUR IDE
# ----------------------------------------------------------------------

def main():
    """Main function to run the SKOS inconsistency detection."""
    print(f"Analyzing file: {INPUT_FILE}")
    inconsistencies = detect_skos_inconsistencies(INPUT_FILE, format=FILE_FORMAT)
    print_inconsistency_report(inconsistencies, OUTPUT_FILE)
    
    # Return the results for further processing if needed
    return inconsistencies

# This allows you to run the script directly in your IDE
if __name__ == '__main__':
    # Uncomment the next lines to run with the test data instead of your own file
    # test_file = create_test_graph()
    # INPUT_FILE = test_file
    
    results = main()

Analyzing file: soil_health_KG.ttl
No inconsistencies found.


In [11]:
def load_rdf_file(file_path):
    """Load an RDF file into a graph."""
    g = Graph()
    
    try:
        g.parse(file_path)
        print(f"Successfully loaded {file_path}")
        print(f"Graph contains {len(g)} triples")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
    
    return g

def extract_uri_relationship_graph(g):
    """Extract a graph that only contains relationships between URIs.
    
    Ignores all triples with literals or blank nodes.
    Returns a dictionary mapping subjects to objects to lists of predicates.
    """
    # Store relationships between concepts (URIs only)
    relationships = defaultdict(lambda: defaultdict(list))
    
    # Count of triples with different node types
    stats = {
        'uri_to_uri': 0,
        'with_blank_nodes': 0,
        'with_literals': 0
    }
    
    # Extract all triples where both subject and object are URIs
    for s, p, o in g:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            # Convert to string and store the predicate
            relationships[str(s)][str(o)].append(str(p))
            stats['uri_to_uri'] += 1
        elif isinstance(s, BNode) or isinstance(o, BNode):
            stats['with_blank_nodes'] += 1
        elif isinstance(o, Literal):
            stats['with_literals'] += 1
    
    print(f"URI-to-URI relationships: {stats['uri_to_uri']}")
    print(f"Relationships with blank nodes: {stats['with_blank_nodes']}")
    print(f"Relationships with literals: {stats['with_literals']}")
    
    return relationships

def detect_multiple_relationships(relationships):
    """Detect cases where more than one predicate links the same two concepts."""
    inconsistencies = []
    
    for subject, objects in relationships.items():
        for object, predicates in objects.items():
            if len(predicates) > 1:
                # Get shortened versions of URIs for display
                short_subject = shorten_uri(subject)
                short_object = shorten_uri(object)
                short_predicates = [shorten_uri(p) for p in predicates]
                
                inconsistencies.append({
                    'type': 'multiple_relationships',
                    'subject': subject,
                    'object': object,
                    'predicates': predicates,
                    'short_subject': short_subject,
                    'short_object': short_object,
                    'short_predicates': short_predicates,
                    'description': f"Multiple relationships between {short_subject} and {short_object}: {', '.join(short_predicates)}"
                })
    
    return inconsistencies

def detect_same_inverse_relationships(relationships):
    """Detect cases where the same predicate is used in both directions between concepts.
    
    Excludes predicates that are naturally symmetric:
    - skos:related (http://www.w3.org/2004/02/skos/core#related)
    - sio:equivalentTo (http://semanticscience.org/resource/equivalentTo)
    - wdt:P1889 (http://www.wikidata.org/prop/direct/P1889)
    - obo:RO_0002434 (http://purl.obolibrary.org/obo/RO_0002434)
    """
    # Define symmetric predicates to ignore
    symmetric_predicates = {
        "http://www.w3.org/2004/02/skos/core#related",
        "http://semanticscience.org/resource/equivalentTo",
        "http://www.wikidata.org/prop/direct/P1889",
        "http://purl.obolibrary.org/obo/RO_0002434"
    }
    
    inconsistencies = []
    processed_pairs = set()  # To avoid reporting the same pair twice
    
    for subject, objects in relationships.items():
        for object, predicates in objects.items():
            # Create a unique identifier for this pair (order-independent)
            pair_id = tuple(sorted([subject, object]))
            
            # Skip if we've already processed this pair
            if pair_id in processed_pairs:
                continue
            
            # Check if the object has relationships back to the subject
            if object in relationships and subject in relationships[object]:
                # Get the predicates in the reverse direction
                reverse_predicates = relationships[object][subject]
                
                # Check for common predicates (excluding symmetric ones)
                common_predicates = set(predicates).intersection(set(reverse_predicates)) - symmetric_predicates
                
                if common_predicates:
                    # Get shortened versions of URIs for display
                    short_subject = shorten_uri(subject)
                    short_object = shorten_uri(object)
                    short_predicates = [shorten_uri(p) for p in common_predicates]
                    
                    inconsistencies.append({
                        'type': 'same_inverse_relationships',
                        'subject': subject,
                        'object': object,
                        'predicates': list(common_predicates),
                        'short_subject': short_subject,
                        'short_object': short_object,
                        'short_predicates': short_predicates,
                        'description': f"Same predicate used in both directions between {short_subject} and {short_object}: {', '.join(short_predicates)}"
                    })
                    
                    # Mark this pair as processed
                    processed_pairs.add(pair_id)
    
    return inconsistencies

def shorten_uri(uri):
    """Extract the last part of a URI for more readable output."""
    if '#' in uri:
        return uri.split('#')[-1]
    elif '/' in uri:
        return uri.split('/')[-1]
    else:
        return uri

def suggest_corrections(inconsistencies):
    """Suggest corrections for the detected inconsistencies."""
    corrections = []
    
    for inconsistency in inconsistencies:
        if inconsistency['type'] == 'multiple_relationships':
            subject = inconsistency['short_subject']
            object = inconsistency['short_object']
            predicates = inconsistency['short_predicates']
            
            correction = {
                'type': 'review_multiple_predicates',
                'subject': subject,
                'object': object,
                'predicates': predicates,
                'description': f"Review multiple predicates between {subject} and {object}: {', '.join(predicates)}. "
                               f"Determine if these are redundant or if they represent distinct relationships."
            }
            corrections.append(correction)
        
        elif inconsistency['type'] == 'same_inverse_relationships':
            subject = inconsistency['short_subject']
            object = inconsistency['short_object']
            predicates = inconsistency['short_predicates']
            
            correction = {
                'type': 'review_bidirectional_predicates',
                'subject': subject,
                'object': object,
                'predicates': predicates,
                'description': f"Review bidirectional use of same predicate(s) between {subject} and {object}: {', '.join(predicates)}. "
                               f"Consider using inverse predicates or removing one direction."
            }
            corrections.append(correction)
    
    return corrections

def format_results(results):
    """Format the results for display."""
    output = []
    
    # Count inconsistencies
    multiple_rel_count = len(results['multiple_relationships'])
    same_inverse_count = len(results['same_inverse_relationships'])
    total_inconsistencies = multiple_rel_count + same_inverse_count
    
    output.append(f"General Relationship Inconsistency Report")
    output.append(f"=======================================")
    output.append(f"\nFound {total_inconsistencies} inconsistencies in total:")
    output.append(f"- {multiple_rel_count} multiple relationship inconsistencies")
    output.append(f"- {same_inverse_count} same inverse relationship inconsistencies")
    
    # Multiple relationship inconsistencies details
    if multiple_rel_count > 0:
        output.append(f"\nMultiple Relationship Inconsistency Details:")
        for i, inconsistency in enumerate(results['multiple_relationships']):
            output.append(f"\n--- Multiple Relationship Inconsistency {i+1} ---")
            output.append(f"Subject: {inconsistency['short_subject']}")
            output.append(f"Object: {inconsistency['short_object']}")
            output.append(f"Predicates: {', '.join(inconsistency['short_predicates'])}")
            
            # Find matching correction
            for correction in results['corrections']:
                if (correction['type'] == 'review_multiple_predicates' and
                    correction['subject'] == inconsistency['short_subject'] and
                    correction['object'] == inconsistency['short_object']):
                    output.append(f"Suggested correction: {correction['description']}")
                    break
    
    # Same inverse relationship inconsistencies details
    if same_inverse_count > 0:
        output.append(f"\nSame Inverse Relationship Inconsistency Details:")
        for i, inconsistency in enumerate(results['same_inverse_relationships']):
            output.append(f"\n--- Same Inverse Relationship Inconsistency {i+1} ---")
            output.append(f"Subject: {inconsistency['short_subject']}")
            output.append(f"Object: {inconsistency['short_object']}")
            output.append(f"Predicates: {', '.join(inconsistency['short_predicates'])}")
            
            # Find matching correction
            for correction in results['corrections']:
                if (correction['type'] == 'review_bidirectional_predicates' and
                    correction['subject'] == inconsistency['short_subject'] and
                    correction['object'] == inconsistency['short_object']):
                    output.append(f"Suggested correction: {correction['description']}")
                    break
    
    return "\n".join(output)

def main(file_path):
    """Main function to run the relationship inconsistency detection."""
    # Step 1: Load RDF file
    g = load_rdf_file(file_path)
    
    # Step 2: Extract URI relationship graph
    relationships = extract_uri_relationship_graph(g)
    relationship_count = sum(len(objects) for objects in relationships.values())
    print(f"Extracted relationship graph has {len(relationships)} subjects and {relationship_count} relationships")
    
    # Step 3: Detect relationship inconsistencies
    multiple_relationships = detect_multiple_relationships(relationships)
    same_inverse_relationships = detect_same_inverse_relationships(relationships)
    
    # Step 4: Suggest corrections
    all_inconsistencies = multiple_relationships + same_inverse_relationships
    corrections = suggest_corrections(all_inconsistencies)
    
    # Prepare results
    results = {
        'multiple_relationships': multiple_relationships,
        'same_inverse_relationships': same_inverse_relationships,
        'corrections': corrections
    }
    
    # Print results
    print(format_results(results))
    
    return results

if __name__ == "__main__":
    # Example usage with a single merged knowledge graph file
    file_path = "soil_health_KG.ttl"  # Replace with your actual file path
    
    results = main(file_path)

Successfully loaded soil_health_KG.ttl
Graph contains 11715 triples
URI-to-URI relationships: 7464
Relationships with blank nodes: 1275
Relationships with literals: 2976
Extracted relationship graph has 2019 subjects and 7434 relationships
General Relationship Inconsistency Report

Found 29 inconsistencies in total:
- 27 multiple relationship inconsistencies
- 2 same inverse relationship inconsistencies

Multiple Relationship Inconsistency Details:

--- Multiple Relationship Inconsistency 1 ---
Subject: SoilErosion
Object: SoilThreats
Predicates: OAE_0000186, indicates
Suggested correction: Review multiple predicates between SoilErosion and SoilThreats: OAE_0000186, indicates. Determine if these are redundant or if they represent distinct relationships.

--- Multiple Relationship Inconsistency 2 ---
Subject: SoilCompaction
Object: SoilBiologicalFunctions
Predicates: RO_0002434, affects
Suggested correction: Review multiple predicates between SoilCompaction and SoilBiologicalFunctions: 

In [13]:
def shorten_uri(uri):
    """Extract the last part of a URI for more readable output."""
    if '#' in uri:
        return uri.split('#')[-1]
    elif '/' in uri:
        return uri.split('/')[-1]
    else:
        return uri

def load_property_mapping(csv_file_path):
    """Load property mapping from CSV file to get predicate-inverse relationships."""
    
    try:
        df = pd.read_csv(csv_file_path)
        print(f"Successfully loaded property mapping from {csv_file_path}")
        print(f"Found {len(df)} predicate-inverse pairs")
        
        # Create bidirectional mapping
        mapping = {}
        for _, row in df.iterrows():
            prop = row['prop']
            inverse = row['inverse']
            
            # Add both directions to the mapping
            mapping[prop] = inverse
            mapping[inverse] = prop
            
        return mapping
    except Exception as e:
        print(f"Error loading property mapping from {csv_file_path}: {e}")
        return {}

def detect_inverse_predicate_conflicts(rdf_file_path, prop_mapping_file_path):
    """
    Detect conflicts where entity pairs have both a predicate and its inverse predicate 
    in the SAME DIRECTION (not opposite directions).
    
    For example, this is a conflict:
    - EntityA --[predicate]--> EntityB AND EntityA --[inverse_predicate]--> EntityB
    
    This is NOT a conflict (it's correct usage):
    - EntityA --[predicate]--> EntityB AND EntityB --[inverse_predicate]--> EntityA
    
    Args:
        rdf_file_path: Path to the RDF file (soil_health_KG.ttl)
        prop_mapping_file_path: Path to the property mapping CSV file
    
    Returns:
        List of conflict reports
    """
    # Load the RDF graph
    g = Graph()
    try:
        g.parse(rdf_file_path)
        print(f"Successfully loaded RDF file: {rdf_file_path}")
        print(f"Graph contains {len(g)} triples")
    except Exception as e:
        print(f"Error loading RDF file {rdf_file_path}: {e}")
        return []
    
    # Load property mapping
    prop_mapping = load_property_mapping(prop_mapping_file_path)
    if not prop_mapping:
        print("No property mapping loaded. Cannot detect inverse conflicts.")
        return []
    
    # Extract all relationships between URI entities
    relationships = defaultdict(lambda: defaultdict(set))
    
    # Store all URI-to-URI relationships
    for s, p, o in g:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            subject_str = str(s)
            predicate_str = str(p)
            object_str = str(o)
            
            relationships[subject_str][object_str].add(predicate_str)
    
    print(f"Extracted {sum(len(objects) for objects in relationships.values())} URI-to-URI relationships")
    
    # Detect conflicts
    conflicts = []
    
    for subject, objects in relationships.items():
        for object_uri, predicates in objects.items():
            # Check if any predicates in the same direction are inverses of each other
            conflict_pairs = []
            
            predicates_list = list(predicates)
            for i, pred1 in enumerate(predicates_list):
                for j, pred2 in enumerate(predicates_list):
                    if i >= j:  # Avoid checking the same pair twice and self-comparison
                        continue
                    
                    # Check if pred1 and pred2 are inverses of each other
                    if pred1 in prop_mapping and prop_mapping[pred1] == pred2:
                        # Skip self-inverse predicates (like skos:related)
                        if pred1 == pred2:
                            continue
                        
                        conflict_pairs.append((pred1, pred2))
            
            if conflict_pairs:
                # Shorten URIs for better readability
                short_subject = shorten_uri(subject)
                short_object = shorten_uri(object_uri)
                
                conflict_details = []
                for pred1, pred2 in conflict_pairs:
                    short_pred1 = shorten_uri(pred1)
                    short_pred2 = shorten_uri(pred2)
                    conflict_details.append({
                        'predicate1': pred1,
                        'predicate2': pred2,
                        'short_predicate1': short_pred1,
                        'short_predicate2': short_pred2
                    })
                
                conflicts.append({
                    'type': 'same_direction_inverse_conflict',
                    'subject': subject,
                    'object': object_uri,
                    'short_subject': short_subject,
                    'short_object': short_object,
                    'conflicts': conflict_details,
                    'description': f"Same direction inverse predicate conflict: {short_subject} -> {short_object}"
                })
    
    return conflicts

def format_inverse_conflict_results(conflicts):
    """Format the inverse predicate conflict results for display."""
    output = []
    
    output.append(f"Same Direction Inverse Predicate Conflict Detection Report")
    output.append(f"=========================================================")
    output.append(f"\nAnalyzing conflicts where entity pairs have both a predicate and its inverse")
    output.append(f"in the SAME DIRECTION (which is a logical contradiction)...")
    output.append(f"Found {len(conflicts)} same direction inverse predicate conflicts:")
    
    if len(conflicts) == 0:
        output.append("\nNo same direction inverse predicate conflicts detected in the knowledge graph.")
        output.append("This means the knowledge graph correctly uses inverse predicates in opposite directions.")
        return "\n".join(output)
    
    for i, conflict in enumerate(conflicts, 1):
        output.append(f"\n--- Conflict {i} ---")
        output.append(f"Entity Pair: {conflict['short_subject']} -> {conflict['short_object']}")
        output.append(f"Full URIs: {conflict['subject']} -> {conflict['object']}")
        output.append(f"Issue: Both predicates go in the same direction but are inverses of each other")
        
        for j, conflict_detail in enumerate(conflict['conflicts'], 1):
            output.append(f"  Conflicting Predicate Pair {i}.{j}:")
            output.append(f"    {conflict['short_subject']} --[{conflict_detail['short_predicate1']}]--> {conflict['short_object']}")
            output.append(f"    {conflict['short_subject']} --[{conflict_detail['short_predicate2']}]--> {conflict['short_object']}")
            output.append(f"    Problem: '{conflict_detail['short_predicate1']}' and '{conflict_detail['short_predicate2']}' are inverse predicates")
            output.append(f"    Full predicate URIs:")
            output.append(f"      {conflict_detail['predicate1']}")
            output.append(f"      {conflict_detail['predicate2']}")
        
        output.append(f"  Recommendation: Remove one of the conflicting predicates, or if both relationships")
        output.append(f"                  are valid, use the inverse predicate in the reverse direction.")
        output.append(f"                  Example: Keep '{conflict['short_subject']} --[predicate]--> {conflict['short_object']}'")
        output.append(f"                          and use '{conflict['short_object']} --[inverse_predicate]--> {conflict['short_subject']}'")
    
    return "\n".join(output)

def run_inverse_conflict_detection(rdf_file_path="soil_health_KG.ttl", 
                                  prop_mapping_file_path="in_ontovocabs/prop_mapping.csv"):
    """
    Main function to run inverse predicate conflict detection.
    
    Args:
        rdf_file_path: Path to the RDF file containing the soil health KG
        prop_mapping_file_path: Path to the property mapping CSV file
    """
    print("Starting same direction inverse predicate conflict detection...")
    print(f"RDF file: {rdf_file_path}")
    print(f"Property mapping file: {prop_mapping_file_path}")
    print("-" * 60)
    
    # Detect conflicts
    conflicts = detect_inverse_predicate_conflicts(rdf_file_path, prop_mapping_file_path)
    
    # Format and display results
    report = format_inverse_conflict_results(conflicts)
    print(report)
    
    return conflicts

# Run the inverse conflict detection
if __name__ == "__main__":
    conflicts = run_inverse_conflict_detection()

Starting same direction inverse predicate conflict detection...
RDF file: soil_health_KG.ttl
Property mapping file: in_ontovocabs/prop_mapping.csv
------------------------------------------------------------
Successfully loaded RDF file: soil_health_KG.ttl
Graph contains 11715 triples
Successfully loaded property mapping from in_ontovocabs/prop_mapping.csv
Found 60 predicate-inverse pairs
Extracted 7434 URI-to-URI relationships
Same Direction Inverse Predicate Conflict Detection Report

Analyzing conflicts where entity pairs have both a predicate and its inverse
in the SAME DIRECTION (which is a logical contradiction)...
Found 0 same direction inverse predicate conflicts:

No same direction inverse predicate conflicts detected in the knowledge graph.
This means the knowledge graph correctly uses inverse predicates in opposite directions.


#### Add metadata of references

In [ ]:
# Merge reference metadata into the processed graphs; the gold standard is untouched.

def merge_reference_metadata(metadata_path, target_paths):
    """Validate both graph unions, then add only missing metadata triples."""
    metadata = Graph().parse(metadata_path, format="turtle")
    references = set(metadata.subjects(RDF.type, DCTERMS.BibliographicResource))
    if not references or set(metadata.subjects()) != references:
        raise ValueError("Metadata must describe existing bibliographic resources only.")
    if any(isinstance(term, BNode) for triple in metadata for term in triple):
        raise ValueError("Metadata must not contain blank nodes.")

    she = Namespace("https://soilwise-he.github.io/soil-health#")
    # Reviewed, compatible title variants; preserve the existing language tags.
    reviewed_titles = {
        she.WRB: {
            Literal("World Reference Base for Soil Resources", lang="en"),
            Literal("World Reference Base for Soil Resources"),
        },
        she.FAOITPS2015: {
            Literal("Status of the world's soil resources", lang="en"),
            Literal("Status of the world's soil resources (SWSR)"),
        },
    }
    # Citation years (date) and verified publication dates (issued) stay separate.
    checked_fields = (DCTERMS.title, DCTERMS.date, DCTERMS.issued,
                      DCTERMS.identifier, DCTERMS.publisher, DCTERMS.isPartOf)
    prepared = []
    for target in map(Path, target_paths):
        original = target.read_bytes()
        graph = Graph().parse(data=original.decode("utf-8-sig"), format="turtle")
        if set(graph.subjects(RDF.type, DCTERMS.BibliographicResource)) != references:
            raise ValueError(f"{target.name}: publication URIs differ from the metadata.")
        for subject in sorted(references):
            for predicate in checked_fields:
                values = set(graph.objects(subject, predicate)) | set(metadata.objects(subject, predicate))
                reviewed = predicate == DCTERMS.title and values <= reviewed_titles.get(subject, set())
                if len(values) > 1 and not reviewed:
                    raise ValueError(f"{target.name}: conflicting {predicate} values for {subject}.")
            identifiers = set(graph.objects(subject, DCTERMS.identifier)) | set(metadata.objects(subject, DCTERMS.identifier))
            sources = set(graph.objects(subject, DCTERMS.source)) | set(metadata.objects(subject, DCTERMS.source))
            if identifiers and sources:
                raise ValueError(f"{target.name}: both identifier and source for {subject}.")

        additions = Graph()
        for prefix, namespace in metadata.namespaces():
            additions.bind(prefix, namespace)
        for triple in metadata:
            if triple not in graph:
                additions.add(triple)
        candidate = original
        if additions:
            # Append the delta to preserve existing formatting and blank-node labels.
            candidate += b"\n\n# Bibliographic metadata from bibliographic_resources_metadata.ttl\n"
            candidate += additions.serialize(format="turtle", encoding="utf-8")
            merged = Graph().parse(data=candidate.decode("utf-8-sig"), format="turtle")
            if not isomorphic(merged, graph + metadata):
                raise ValueError(f"{target.name}: serialized union failed validation.")
        prepared.append((target, original, candidate, len(graph), len(additions)))

    # Validate both outputs before replacing either file; a rerun is a byte-level no-op.
    staged = []
    try:
        for target, original, candidate, _, added in prepared:
            if target.read_bytes() != original:
                raise RuntimeError(f"{target.name} changed during validation; rerun the cell.")
            if added:
                with NamedTemporaryFile(dir=target.parent, prefix=target.name + ".", suffix=".tmp", delete=False) as handle:
                    temporary = Path(handle.name)
                    staged.append((target, temporary))
                    handle.write(candidate)
        for target, temporary in staged:
            temporary.replace(target)
    finally:
        for _, temporary in staged:
            temporary.unlink(missing_ok=True)

    print(f"Metadata: {len(references)} publications, {len(metadata):,} triples; conflict checks passed.")
    for target, _, _, before, added in prepared:
        print(f"{target.name}: {before:,} -> {before + added:,} triples (+{added:,}); {len(references)} publications.")


merge_reference_metadata(
    "bibliographic_resources_metadata.ttl",
    ("soil_health_KG.ttl", "soil_health_SKOS.ttl"),
)


Metadata: 155 publications, 1,644 triples; conflict checks passed.
soil_health_KG.ttl: 11,712 -> 13,052 triples (+1,340); 155 publications.
soil_health_SKOS.ttl: 16,778 -> 18,118 triples (+1,340); 155 publications.


### Usecase #1: SoilWise knowledge repository (demo)

In [241]:
# Define the API endpoint URL
api_url = "https://zenodo.org/api/records"

# list to gather all data through API requests
zenodo_data = []

# loop through pages to retreive next part of data
for i in range(1,11):
    print(f'loop num {i}')
    params = {'q': 'soil',
              'size':'50',
              'page':f'{i}',
              'status':'published',
              'sort':'bestmatch',
              'access_token': config['access_token_zenodo']}
    headers = {"Content-Type": "application/json"}

    # Make the API request
    response = requests.get(api_url,params = params, headers=headers)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()
        zenodo_data = zenodo_data + data['hits']['hits']
    else:
        print(f"Failed to retrieve data. Status code: {response.status_code}")
    
zenodo_data

loop num 1
loop num 2
loop num 3
loop num 4
loop num 5
loop num 6
loop num 7
loop num 8
loop num 9
loop num 10


[{'created': '2021-08-05T16:06:00.842433+00:00',
  'modified': '2025-05-19T15:23:11.635528+00:00',
  'id': 5159133,
  'conceptrecid': '5159132',
  'doi': '10.5281/zenodo.5159133',
  'conceptdoi': '10.5281/zenodo.5159132',
  'doi_url': 'https://doi.org/10.5281/zenodo.5159133',
  'metadata': {'title': 'Soil Survey of Scotland Staff (1970-1987). Soil maps of Scotland (partial coverage) at a scale of 1:25 000. Digital phase 8 release.',
   'doi': '10.5281/zenodo.5159133',
   'publication_date': '2021-08-04',
   'description': '<p>This is the digital dataset which was created by digitising the Soils of Scotland 1:25,000 Soil maps and the Soils of Scotland 1:25,000 Dyeline Masters. The Soils of Scotland 1:25,000 Soil maps were the source documents for the production of the Soils of Scotland 1:63,360 and 1:50,000 published map series. Where no 1:25,000 published maps exist 1:63,360 maps have been digitised for this data set, the field SOURCE_MAP describes the source of the data. The classific

In [242]:
# gather the id's in list to then extract data directly as dublincore representation
id_list = []
for i in range(0,len(zenodo_data)):
    id_list.append(zenodo_data[i]['id'])

In [243]:
# Convert HTML to plain text
class CleanHTML(HTMLParser):
    def reset(self) -> None:
        self.extracted_data = ""
        return super().reset()

    def remove_tags(self, html_data: str) -> str:
        """
        Args:
            html_data (str): HTML data which might contain tags.

        Returns:
            str: Data without any HTML tags. Forces feeding of any buffered data.
        """
        self.reset()
        self.feed(html_data)
        self.close()
        return self.extracted_data

    def handle_data(self, data: str) -> None:
        """
        Args:
            data (str): Html data extracted from tags to be processed.
        """
        self.extracted_data += data

p = CleanHTML()

In [244]:
# Extract DublinCore standardized metadata and generate json with all data
zenodo_dc_data = {}
for i in range(len(id_list)):

    # Define the API endpoint URL
    api_url = f"https://zenodo.org/records/{id_list[i]}/export/dublincore"
    params = {'access_token': config['access_token_zenodo']}
    headers = {"Content-Type": "application/json"}

    # Make the API request
    response = requests.get(api_url, params = params, headers=headers, timeout=None)

    # Check if the request was successful
    if response.status_code == 200:
        try:
            # Parse the XML response
            root = ET.fromstring(response.content)

            # Define expected tags with namespace
            expected_tags = {
                "{http://purl.org/dc/elements/1.1/}creator": None,      # Initialize as None
                "{http://purl.org/dc/elements/1.1/}subject": None,      # Initialize as None
                "{http://purl.org/dc/elements/1.1/}date": None,         # Initialize as None
                "{http://purl.org/dc/elements/1.1/}title": None,        # Initialize as None
                "{http://purl.org/dc/elements/1.1/}description": None,   # Initialize as None
                "{http://purl.org/dc/elements/1.1/}identifier": None,    # Initialize as None
                "{http://purl.org/dc/elements/1.1/}language": None,    # Initialize as None
                "{http://purl.org/dc/elements/1.1/}type": None    # Initialize as None
            }

            # Create a dictionary for JSON conversion
            json_dict = {}

            # Check if the root has child elements
            if root:
                for child in root:
                    # Get full tag name, preserving namespace
                    tag = child.tag
                    text = ftfy.fix_text(p.remove_tags(unescape(child.text)))  # text sanitization

                    if tag in expected_tags:
                        if "creator" in tag or "subject" in tag:
                            if "creator" in tag:
                                if json_dict.get(tag) is None:
                                    json_dict[tag] = [text]
                                else:
                                    json_dict[tag].append(text)
                            elif "subject" in tag:
                                # Split subjects and strip spaces                              
                                subjects = [subj.strip() for subj in re.split(r'[;,]', text)]
                                if json_dict.get(tag) is None:                                  
                                    json_dict[tag] = subjects
                                else:
                                    json_dict[tag].extend(subjects)
                        else:
                            if json_dict.get(tag) is None:
                                json_dict[tag] = text  # Directly store as a string for date, title, description and identifier

                # Ensure all expected tags are included in the json_dict
                for tag, value in expected_tags.items():
                    if tag not in json_dict:
                        json_dict[tag] = value

                output_dict = {}
                # Adjusting the output to match the "dc:" prefix requirement
                for key, value in json_dict.items():
                    adjusted_key = key.split('}')[-1]  # Remove namespace URI
                    output_dict["dc:" + adjusted_key] = value

                zenodo_dict={
                    i:output_dict
                }
                
                #add to general json
                zenodo_dc_data.update(zenodo_dict)
            
            else:
                print("No repositories found in the response.")
        except ET.ParseError:
            print("Error: Unable to parse XML content.")

zenodo_dc_data

{0: {'dc:creator': ['Soil Survey Staff'],
  'dc:date': '2021-08-04',
  'dc:description': "This is the digital dataset which was created by digitising the Soils of Scotland 1:25,000 Soil maps and the Soils of Scotland 1:25,000 Dyeline Masters. The Soils of Scotland 1:25,000 Soil maps were the source documents for the production of the Soils of Scotland 1:63,360 and 1:50,000 published map series. Where no 1:25,000 published maps exist 1:63,360 maps have been digitised for this data set, the field SOURCE_MAP describes the source of the data. The classification is based on Soil Associations, Soil Series and Phases which reflect parent material, major soil group, and soil sub-groups, drainage and (for phases), texture, stoniness, land use, rockiness, topography and organic matter. Phases are not always mapped. In general terms this dataset primarily covers the cultivated land of Scotland but also includes some upland areas.\xa0 This data set is undergoing a phased revision, the latest (phas

In [245]:
# Convert metadata to RDF statements
# Dictionary of metadata entries, indexed by a unique key
metadata_entries = zenodo_dc_data

# Fill in all the missing keywords under named graph "augmented"
g_zenodo_extend = Dataset()

# Define named graphs
metadata_graph = URIRef("https://soilwise-he.github.io/soil-health#metadata")
generated_graph = URIRef("https://soilwise-he.github.io/soil-health#augmented")

# keywords generation
def generate_subjects(text):
    # Placeholder function to simulate generating subjects based on the title
    kw_model = KeyBERT()
    seed_keywords = ["soil"]

    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None, seed_keywords=seed_keywords, use_mmr=True, diversity=0.5)
    keywords = [item[0] for item in keywords if item[1] > 0.5]
    
    return keywords

# Loop through each metadata entry
for entry_id, metadata in metadata_entries.items():
    if metadata['dc:language'] == 'eng': 
        # Define the URI for the resource using its identifier
        resource_uri = URIRef(metadata['dc:identifier'])
    
        # Add the title, date, and identifier
        g_zenodo_extend.add((resource_uri, DCTERMS.title, Literal(metadata['dc:title']), metadata_graph))
        g_zenodo_extend.add((resource_uri, DCTERMS.date, Literal(metadata['dc:date']), metadata_graph))
        g_zenodo_extend.add((resource_uri, DCTERMS.identifier, URIRef(metadata['dc:identifier']), metadata_graph))
        g_zenodo_extend.add((resource_uri, DCTERMS.type, URIRef(metadata['dc:type']), metadata_graph))

        # Add creators, which can have multiple values
        for creator in metadata['dc:creator']:
            g_zenodo_extend.add((resource_uri, DCTERMS.creator, Literal(creator), metadata_graph))
    
        # Check if subjects are provided, otherwise generate them
        if metadata['dc:subject'] is None:
            generated_subjects = generate_subjects(f"{metadata['dc:title']}. {metadata['dc:description']}")
            for subject in generated_subjects:
                g_zenodo_extend.add((resource_uri, DCTERMS.subject, Literal(subject), generated_graph))
        else:
            for subject in metadata['dc:subject']:
                # Strip whitespace, remove trailing periods, normalize spaces
                cleaned_subject = re.sub(r'\s+', ' ', subject.strip().rstrip('.').strip())
                g_zenodo_extend.add((resource_uri, DCTERMS.subject, Literal(cleaned_subject), metadata_graph))

# Output the graph in Turtle format
g_zenodo_extend.bind("she", she)

print(g_zenodo_extend.serialize(format="trig"))

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix she: <https://soilwise-he.github.io/soil-health#> .

she:metadata {
    <https://doi.org/10.1007/s11119-021-09805-y> dcterms:creator "Blagojević, Dragana",
            "Dedović, Nebojša",
            "Ivošević, Bojana",
            "Kostić, Marko",
            "Ljubičić, Nataša",
            "Radulović, Mirjana",
            "Rajković, Miloš" ;
        dcterms:date "2021-04-11" ;
        dcterms:identifier <https://doi.org/10.1007/s11119-021-09805-y> ;
        dcterms:subject "Central composite design",
            "On-the-go measurement",
            "Sensors",
            "Soil properties" ;
        dcterms:title "Georeferenced tractor wheel slip data for prediction of spatial variability in soil physical properties" ;
        dcterms:type <info:eu-repo/semantics/article> .

    <https://doi.org/10.1016/j.biocon.2022.109475> dcterms:creator "Briones, M.J.I.",
            "Jones, A.",
            "Köninger, J.",
            "Orgia

In [ ]:
# Interlink metadata quads with soil health knowledge graph
# Function to link concepts
def link_concepts(graph, concept_graph):

    AGRONTOLOGY = Namespace("http://aims.fao.org/aos/agrontology#")

    # Extract all concept URIs and labels
    labels = {}
    for c in concept_graph.subjects(RDF.type, None):
        label_list = []
        
        # Get prefLabel
        pref_label = concept_graph.value(c, SKOS.prefLabel)
        if pref_label:
            label_list.append(pref_label.lower())
        
        # Get all altLabels
        for alt_label in concept_graph.objects(c, SKOS.altLabel):
            if alt_label:
                label_list.append(alt_label.lower())
        
        # Get all abbreviations
        for abbrev in concept_graph.objects(c, AGRONTOLOGY.hasAbbreviation):
            if abbrev:
                label_list.append(abbrev.lower())
        
        # Get all symbols
        for symbol in concept_graph.objects(c, AGRONTOLOGY.hasSymbol):
            if symbol:
                label_list.append(symbol.lower())
        
        labels[c] = label_list

    for s, p, o, g in graph.quads((None, None, None, None)):
        subject_label = o.lower()
        for c, label_list in labels.items():
            if subject_label in label_list:
                graph.remove((s, p, o, g))
                graph.add((s, p, c, g))  # Preserve original predicate
                concept_graph.add((c, DCTERMS.source, s))

# Link concepts between metadata and concept graphs
link_concepts(g_zenodo_extend, unique_graph)

# Save the augmented graph to a file
g_zenodo_extend.serialize(destination="example_SWR.trig", format="trig")

<Graph identifier=N11b4638aa04946eab2f36dfa2b4cff54 (<class 'rdflib.graph.Dataset'>)>

### KG validation

In [4]:
with open('benchmarks/CQs_SPARQL_ea.json', 'r') as f:
    qa = json.load(f)
query = qa['qa_pairs'][1]['sparql_query']

print(query)

PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX agrontology: <http://aims.fao.org/aos/agrontology#>

SELECT ?subjectLabel ?predicateLabel ?objectLabel
WHERE {
  VALUES ?practiceLabelString { "sustainable soil management"@en "soil protection"@en }
  ?practice skos:prefLabel ?practiceLabelString .
  
  {
    ?practice obo:T4FS_0000577 ?targetOfPractice .
    ?practice skos:prefLabel ?subjectLabel .
    BIND(obo:T4FS_0000577 AS ?predicateLabel)
    ?targetOfPractice skos:prefLabel ?objectLabel .
  }
  
  UNION
  
  {
    ?practice obo:T4FS_0000577 ?targetOfPractice .
    ?targetOfPractice agrontology:isAffectedBy ?affectedItem .
    ?targetOfPractice skos:prefLabel ?subjectLabel .
    BIND(agrontology:isAffectedBy AS ?predicateLabel)
    ?affectedItem skos:prefLabel ?objectLabel .
  }
}


In [ ]:
def sparql_to_dataframe(graph, query):
    """
    Executes a SPARQL query against an rdflib Graph and returns the results
    as a pandas DataFrame.
    """
    # Execute the query
    results = graph.query(query)

    # Extract variable names from the query results
    variables = [str(var) for var in results.vars]

    # Convert the results to a list of dictionaries
    data = []
    for row in results:
        row_dict = {}
        for var in variables:
            # Get the value for each variable in the current row
            value = getattr(row, var) if hasattr(row, var) else None
            # Convert the value to a string, handling None values
            row_dict[var] = str(value) if value is not None else "" # Use empty string for None to match CSV output
        data.append(row_dict)

    # Create and return the DataFrame
    return pd.DataFrame(data)

# Initialize an empty graph
g = Graph()
g.parse("soil_health_KG.ttl", format="turtle")

df = sparql_to_dataframe(g, query)
print(df.to_csv(index=False, quoting=csv.QUOTE_ALL))

"subjectLabel","predicateLabel","objectLabel"
"state of soils","http://aims.fao.org/aos/agrontology#isAffectedBy","climatic conditions"
"sustainable soil management","http://purl.obolibrary.org/obo/T4FS_0000577","state of soils"
"state of soils","http://aims.fao.org/aos/agrontology#isAffectedBy","soil management"
"soil protection","http://purl.obolibrary.org/obo/T4FS_0000577","critical level at preventive activity"
"soil protection","http://purl.obolibrary.org/obo/T4FS_0000577","critical level at soil functions"
"state of soils","http://aims.fao.org/aos/agrontology#isAffectedBy","climatic conditions"
"soil protection","http://purl.obolibrary.org/obo/T4FS_0000577","state of soils"
"soil protection","http://purl.obolibrary.org/obo/T4FS_0000577","critical level at restorative activity"
"state of soils","http://aims.fao.org/aos/agrontology#isAffectedBy","soil management"



In [ ]:
def get_label_from_kg(uri, graph):
    """
    Get the skos:prefLabel for a URI from the knowledge graph.
    Only for URIs starting with 'https://soilwise-he.github.io/soil-health#'
    and where the first letter after '#' is uppercase.
    """
    if not isinstance(uri, str):
        return None
    
    # Check if URI matches the pattern
    if uri.startswith("https://soilwise-he.github.io/soil-health#"):
        fragment = uri.split("#")[-1]
        if fragment and fragment[0].isupper():
            # Query the graph for skos:prefLabel
            uri_ref = URIRef(uri)
            for _, _, label in graph.triples((uri_ref, SKOS.prefLabel, None)):
                return str(label)
    return None

def get_label_from_csv(uri, properties_df):
    """
    Get the label for a URI from properties.csv.
    URI is in the second column (index 1), label in the third column (index 2).
    """
    if not isinstance(uri, str):
        return None
    
    # Find the row with matching URI
    match = properties_df[properties_df['uri'] == uri]
    if not match.empty:
        return match.iloc[0]['label']
    return None

def uri_to_label(value, kg_graph, properties_df):
    """
    Convert a URI to its human-readable label.
    First tries to get label from KG (for soil-health URIs with uppercase after #),
    then tries properties.csv, otherwise returns the original value.
    """
    if not isinstance(value, str) or not value.startswith("http"):
        return value
    
    # Try to get label from KG first
    label = get_label_from_kg(value, kg_graph)
    if label:
        return label
    
    # Try to get label from properties.csv
    label = get_label_from_csv(value, properties_df)
    if label:
        return label
    
    # If no label found, return the original URI
    return value

def row_to_sentence(row_dict, column_names):
    """
    Convert a row with any number of columns into a natural language sentence.
    Simply joins the values with spaces, ignoring empty values.
    
    Parameters:
    - row_dict: Dictionary of column_name -> value
    - column_names: List of column names in order
    
    Returns:
    - A sentence string
    """
    # Get all values in order, filtering out empty strings
    values = []
    for col in column_names:
        value = str(row_dict[col]).strip()
        if value:  # Only include non-empty values
            values.append(value)
    
    # Join values with spaces and add period at the end
    if values:
        return " ".join(values) + "."
    else:
        return ""

def convert_query_results_to_sentences(df, kg_graph, properties_csv_path='in_ontovocabs/properties.csv', remove_duplicates=True):
    """
    Convert SPARQL query results DataFrame to human-readable sentences.
    Works with any number of columns in the DataFrame.
    Automatically removes duplicate sentences by default.
    
    Parameters:
    - df: DataFrame with query results (can have any number of columns)
    - kg_graph: rdflib Graph loaded with soil_health_KG.ttl
    - properties_csv_path: Path to properties.csv file
    - remove_duplicates: If True (default), removes exact duplicate sentences while preserving order
    
    Returns:
    - List of unique sentences
    """
    # Load properties.csv
    properties_df = pd.read_csv(properties_csv_path)
    
    sentences = []
    cols = df.columns.tolist()
    
    # Iterate through each row in the DataFrame
    for idx, row in df.iterrows():
        # Create a dictionary with replaced labels for all columns
        row_with_labels = {}
        for col in cols:
            value = row[col]
            label = uri_to_label(value, kg_graph, properties_df)
            row_with_labels[col] = label
        
        # Convert to sentence based on number of columns
        sentence = row_to_sentence(row_with_labels, cols)
        sentences.append(sentence)
    
    # Remove duplicates if requested
    if remove_duplicates:
        seen = set()
        unique_sentences = []
        for sentence in sentences:
            if sentence and sentence not in seen:
                seen.add(sentence)
                unique_sentences.append(sentence)
        return unique_sentences
    
    return sentences

# Convert the query results to sentences (duplicates automatically removed)
sentences = convert_query_results_to_sentences(df, g)

# Print the sentences (one per line, no index numbers)
for sentence in sentences:
    if sentence:  # Only print non-empty sentences
        print(sentence)


state of soils is affected by climatic conditions.
sustainable soil management requires knowledge about state of soils.
state of soils is affected by soil management.
soil protection requires knowledge about critical level at preventive activity.
soil protection requires knowledge about critical level at soil functions.
soil protection requires knowledge about state of soils.
soil protection requires knowledge about critical level at restorative activity.


In [ ]:
# Complete workflow: Query -> DataFrame -> Sentences
def query_to_sentences(graph, sparql_query, properties_csv_path='in_ontovocabs/properties.csv', 
                      save_to_file=None, verbose=True):
    """
    Complete workflow to convert a SPARQL query to human-readable sentences.
    
    Parameters:
    - graph: rdflib Graph with the knowledge graph loaded
    - sparql_query: SPARQL query string
    - properties_csv_path: Path to properties.csv
    - save_to_file: Optional filename to save sentences (None = don't save)
    - verbose: Print progress messages
    
    Returns:
    - Tuple: (sentences list, DataFrame with original URIs, DataFrame with labels)
    """
    if verbose:
        print("Step 1: Executing SPARQL query...")
    df_original = sparql_to_dataframe(graph, sparql_query)
    
    if verbose:
        print(f"Step 2: Converting {len(df_original)} rows to sentences...")
    sentences = convert_query_results_to_sentences(df_original, graph, properties_csv_path)
    
    # Create labeled DataFrame for all columns
    properties_df = pd.read_csv(properties_csv_path)
    cols = df_original.columns.tolist()
    df_labeled = pd.DataFrame()
    
    for col in cols:
        df_labeled[col] = df_original[col].apply(lambda x: uri_to_label(x, graph, properties_df))
    
    if verbose:
        print("Step 3: Verifying URI replacements...")
    
    if verbose:
        print(f"\n✓ Successfully converted {len(sentences)} rows to sentences!")
    
    return sentences, df_original, df_labeled

# Example: Run the complete workflow
print("=" * 70)
print("COMPLETE WORKFLOW EXAMPLE")
print("=" * 70)
sentences_result, df_orig, df_lab = query_to_sentences(g, query, verbose=True)

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
for sentence in sentences_result:
    if sentence:
        print(sentence)

COMPLETE WORKFLOW EXAMPLE
Step 1: Executing SPARQL query...
Step 2: Converting 9 rows to sentences...
Step 3: Verifying URI replacements...

✓ Successfully converted 7 rows to sentences!

FINAL RESULTS
state of soils is affected by climatic conditions.
sustainable soil management requires knowledge about state of soils.
state of soils is affected by soil management.
soil protection requires knowledge about critical level at preventive activity.
soil protection requires knowledge about critical level at soil functions.
soil protection requires knowledge about state of soils.
soil protection requires knowledge about critical level at restorative activity.


In [ ]:
# Test with another query from the benchmarks
print("\n" + "=" * 70)
print("TESTING WITH ANOTHER QUERY")
print("=" * 70)

# Load another query from the benchmarks
if len(qa['qa_pairs']) > 1:
    query2 = qa['qa_pairs'][87]['sparql_query']
    print(f"\nQuery text: {qa['qa_pairs'][87].get('competency_question', 'N/A')}")
    print(f"\nSPARQL Query:\n{query2}\n")
    
    sentences2, df_orig2, df_lab2 = query_to_sentences(g, query2, verbose=True)
    
    print("\n" + "=" * 70)
    print("RESULTS")
    print("=" * 70)
    for sentence in sentences2:
        if sentence:
            print(sentence)
else:
    print("No additional queries available in the benchmark file.")


TESTING WITH ANOTHER QUERY

Query text: What is a set of complex soil physical parameters that can document the consequences of soil compaction and soil degradation on physical, chemical and biological functions and what are their symbols and units, if any?

SPARQL Query:
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX she: <https://soilwise-he.github.io/soil-health#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX unit: <http://qudt.org/vocab/unit/>
PREFIX om: <http://www.ontology-of-units-of-measure.org/resource/om-2/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX agrontology: <http://aims.fao.org/aos/agrontology#>

SELECT ?parameterLabel ?propertyURI ?valueLabel
WHERE {
  she:ComplexSoilPhysicalParameters skos:narrower ?parameterURI .
  ?parameterURI skos:prefLabel ?parameterLabel .

  {
    ?parameterURI qudt:unit ?unitValueURI .
    OPTIONAL { ?unitValueURI skos:prefLabel ?unitPrefLabel . }
    BIND(
      COALESCE(
        ?unitPrefLabel,
        IF( STRSTAR

In [ ]:
sparql = SPARQLWrapper("https://repository.soilwise-he.eu/sparql/")
sparql.setQuery(query)
sparql.setReturnFormat(CSV)
results = sparql.queryAndConvert()

print(results.decode('utf-8'))

"subjectLabel","predicateURI","objectLabel"
"soil pollution thresholds","http://purl.obolibrary.org/obo/T4FS_0000577","concentration regulation processes"
"soil pollution thresholds","http://purl.obolibrary.org/obo/T4FS_0000577","pollutant emissions"
"soil pollution thresholds","http://purl.obolibrary.org/obo/T4FS_0000577","pollutant exposure"
"soil pollution thresholds","http://purl.obolibrary.org/obo/T4FS_0000577","soil inputs"
"soil pollution thresholds","http://purl.obolibrary.org/obo/T4FS_0000577","soil outputs"



#### Knowledge Graph Evaluation Against Competency Questions (Token-level F1 score)

In [ ]:
def tokenize(text):
    """
    Tokenize text into words, converting to lowercase and removing punctuation.
    """
    # Convert to lowercase and split into words
    text = text.lower()
    # Remove punctuation and split
    words = re.findall(r'\b\w+\b', text)
    return words

def calculate_token_metrics(predicted_text, expected_text):
    """
    Calculate precision, recall, and F1 score based on token overlap.
    
    Parameters:
    - predicted_text: The text generated from query results
    - expected_text: The ground truth expected answer
    
    Returns:
    - Dictionary with precision, recall, F1, and token counts
    """
    # Tokenize both texts
    predicted_tokens = set(tokenize(predicted_text))
    expected_tokens = set(tokenize(expected_text))
    
    # Calculate overlap
    common_tokens = predicted_tokens.intersection(expected_tokens)
    
    # Calculate metrics
    if len(predicted_tokens) == 0:
        precision = 0.0
    else:
        precision = len(common_tokens) / len(predicted_tokens)
    
    if len(expected_tokens) == 0:
        recall = 0.0
    else:
        recall = len(common_tokens) / len(expected_tokens)
    
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'common_tokens': len(common_tokens),
        'predicted_tokens': len(predicted_tokens),
        'expected_tokens': len(expected_tokens),
        'common_token_list': sorted(list(common_tokens))
    }

def evaluate_single_qa(qa_pair, graph, properties_csv_path='in_ontovocabs/properties.csv', verbose=False):
    """
    Evaluate a single QA pair by running the query and comparing with expected answer.
    
    Parameters:
    - qa_pair: Dictionary with 'competency_question', 'sparql_query', 'expected_answer'
    - graph: rdflib Graph with the knowledge graph
    - properties_csv_path: Path to properties.csv
    - verbose: Print detailed information
    
    Returns:
    - Dictionary with evaluation results
    """
    try:
        # Run the query
        df_result = sparql_to_dataframe(graph, qa_pair['sparql_query'])
        
        # Convert to sentences
        sentences = convert_query_results_to_sentences(df_result, graph, properties_csv_path)
        
        # Combine all sentences into one text
        predicted_answer = ' '.join(sentences)
        
        # Get expected answer
        expected_answer = qa_pair['expected_answer']
        
        # Calculate metrics
        metrics = calculate_token_metrics(predicted_answer, expected_answer)
        
        result = {
            'question': qa_pair['competency_question'],
            'expected_answer': expected_answer,
            'predicted_answer': predicted_answer,
            'num_results': len(df_result),
            'num_sentences': len(sentences),
            **metrics,
            'success': True,
            'error': None
        }
        
        if verbose:
            print(f"Question: {qa_pair['competency_question']}")
            print(f"Expected: {expected_answer}")
            print(f"Predicted: {predicted_answer}")
            print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1']:.3f}")
            print("-" * 80)
        
        return result
        
    except Exception as e:
        return {
            'question': qa_pair['competency_question'],
            'expected_answer': qa_pair.get('expected_answer', ''),
            'predicted_answer': '',
            'num_results': 0,
            'num_sentences': 0,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'common_tokens': 0,
            'predicted_tokens': 0,
            'expected_tokens': 0,
            'common_token_list': [],
            'success': False,
            'error': str(e)
        }

print("✓ Evaluation functions defined successfully!")

✓ Evaluation functions defined successfully!


In [ ]:
def evaluate_all_qa_pairs(json_file_path, graph, properties_csv_path='in_ontovocabs/properties.csv', 
                          max_questions=None, verbose=False):
    """
    Evaluate all QA pairs from the JSON file.
    
    Parameters:
    - json_file_path: Path to the CQs_SPARQL_ea.json file
    - graph: rdflib Graph with the knowledge graph
    - properties_csv_path: Path to properties.csv
    - max_questions: Maximum number of questions to evaluate (None = all)
    - verbose: Print detailed information for each question
    
    Returns:
    - List of evaluation results and summary statistics
    """
    import json
    
    # Load QA pairs
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    qa_pairs = data['qa_pairs']
    
    if max_questions:
        qa_pairs = qa_pairs[:max_questions]
    
    print(f"Evaluating {len(qa_pairs)} competency questions...")
    print("=" * 80)
    
    results = []
    
    for i, qa_pair in enumerate(qa_pairs, 1):
        if verbose:
            print(f"\nQuestion {i}/{len(qa_pairs)}:")
        else:
            if i % 10 == 0:
                print(f"Processed {i}/{len(qa_pairs)} questions...")
        
        result = evaluate_single_qa(qa_pair, graph, properties_csv_path, verbose=verbose)
        result['question_id'] = i
        results.append(result)
    
    print("\n" + "=" * 80)
    print("Evaluation Complete!")
    
    # Calculate summary statistics
    successful_results = [r for r in results if r['success']]
    failed_results = [r for r in results if not r['success']]
    
    if successful_results:
        avg_precision = sum(r['precision'] for r in successful_results) / len(successful_results)
        avg_recall = sum(r['recall'] for r in successful_results) / len(successful_results)
        avg_f1 = sum(r['f1'] for r in successful_results) / len(successful_results)
    else:
        avg_precision = avg_recall = avg_f1 = 0.0
    
    summary = {
        'total_questions': len(qa_pairs),
        'successful_evaluations': len(successful_results),
        'failed_evaluations': len(failed_results),
        'average_precision': avg_precision,
        'average_recall': avg_recall,
        'average_f1': avg_f1,
        'questions_with_perfect_recall': len([r for r in successful_results if r['recall'] == 1.0]),
        'questions_with_perfect_precision': len([r for r in successful_results if r['precision'] == 1.0]),
        'questions_with_perfect_f1': len([r for r in successful_results if r['f1'] == 1.0])
    }
    
    return results, summary

print("✓ Batch evaluation function defined successfully!")

✓ Batch evaluation function defined successfully!


In [ ]:
# Evaluate ALL competency questions
print("=" * 80)
print("EVALUATING ALL COMPETENCY QUESTIONS")
print("=" * 80)

all_results, all_summary = evaluate_all_qa_pairs(
    'benchmarks/CQs_SPARQL_ea.json', 
    g,
    verbose=False  # Set to True to see details for each question
)

print("\n" + "=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)
print(f"\nTotal Questions Evaluated: {all_summary['total_questions']}")
print(f"Successful Evaluations: {all_summary['successful_evaluations']}")
print(f"Failed Evaluations: {all_summary['failed_evaluations']}")

print(f"\n{'Metric':<20} {'Score':<10} {'Interpretation'}")
print("-" * 80)
print(f"{'Average Precision:':<20} {all_summary['average_precision']:<10.4f} Proportion of predicted tokens that are correct")
print(f"{'Average Recall:':<20} {all_summary['average_recall']:<10.4f} Proportion of expected tokens that were found")
print(f"{'Average F1 Score:':<20} {all_summary['average_f1']:<10.4f} Harmonic mean of precision and recall")

print(f"\n{'Perfect Scores':<40} {'Count':<10} {'Percentage'}")
print("-" * 80)
total = all_summary['successful_evaluations']
print(f"{'Questions with Perfect Recall (1.0):':<40} {all_summary['questions_with_perfect_recall']:<10} {all_summary['questions_with_perfect_recall']/total*100:.1f}%")
print(f"{'Questions with Perfect Precision (1.0):':<40} {all_summary['questions_with_perfect_precision']:<10} {all_summary['questions_with_perfect_precision']/total*100:.1f}%")
print(f"{'Questions with Perfect F1 (1.0):':<40} {all_summary['questions_with_perfect_f1']:<10} {all_summary['questions_with_perfect_f1']/total*100:.1f}%")

EVALUATING ALL COMPETENCY QUESTIONS
Evaluating 105 competency questions...
Processed 10/105 questions...
Processed 10/105 questions...
Processed 20/105 questions...
Processed 20/105 questions...
Processed 30/105 questions...
Processed 30/105 questions...
Processed 40/105 questions...
Processed 40/105 questions...
Processed 50/105 questions...
Processed 50/105 questions...
Processed 60/105 questions...
Processed 60/105 questions...
Processed 70/105 questions...
Processed 70/105 questions...
Processed 80/105 questions...
Processed 80/105 questions...


https://soilwise-he.github.io/soil-health#Egli2014 (2014) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Montgomery2007 (2007) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Montgomery2007 (2007) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Verheijen2009 (2009) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Wakatsuki1992 (1992) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Montgomery2007 (2007) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Montgomery2007 (2007) does not look like a valid URI, trying to serialize this will break.
https://soilwise-he.github.io/soil-health#Verheijen2009 (2009) does not look 

Processed 90/105 questions...
Processed 100/105 questions...
Processed 100/105 questions...

Evaluation Complete!

FINAL EVALUATION SUMMARY

Total Questions Evaluated: 105
Successful Evaluations: 105
Failed Evaluations: 0

Metric               Score      Interpretation
--------------------------------------------------------------------------------
Average Precision:   0.7667     Proportion of predicted tokens that are correct
Average Recall:      0.6305     Proportion of expected tokens that were found
Average F1 Score:    0.6817     Harmonic mean of precision and recall

Perfect Scores                           Count      Percentage
--------------------------------------------------------------------------------
Questions with Perfect Recall (1.0):     3          2.9%
Questions with Perfect Precision (1.0):  6          5.7%
Questions with Perfect F1 (1.0):         1          1.0%

Evaluation Complete!

FINAL EVALUATION SUMMARY

Total Questions Evaluated: 105
Successful Evaluations: 1

In [ ]:
# Create a comprehensive results DataFrame
results_df = pd.DataFrame(all_results)

# Display statistics
print("\n" + "=" * 80)
print("SCORE DISTRIBUTION")
print("=" * 80)
print("\nPrecision Statistics:")
print(results_df['precision'].describe())
print("\nRecall Statistics:")
print(results_df['recall'].describe())
print("\nF1 Score Statistics:")
print(results_df['f1'].describe())

# Show distribution by score ranges
print("\n" + "=" * 80)
print("SCORE RANGE DISTRIBUTION")
print("=" * 80)

def categorize_score(score):
    if score >= 0.9:
        return "Excellent (0.9-1.0)"
    elif score >= 0.7:
        return "Good (0.7-0.9)"
    elif score >= 0.5:
        return "Fair (0.5-0.7)"
    elif score >= 0.3:
        return "Poor (0.3-0.5)"
    else:
        return "Very Poor (0.0-0.3)"

results_df['f1_category'] = results_df['f1'].apply(categorize_score)

print("\nF1 Score Distribution:")
print(results_df['f1_category'].value_counts().sort_index())

# Show token coverage
print("\n" + "=" * 80)
print("TOKEN COVERAGE ANALYSIS")
print("=" * 80)
print(f"Total expected tokens across all questions: {results_df['expected_tokens'].sum()}")
print(f"Total predicted tokens across all questions: {results_df['predicted_tokens'].sum()}")
print(f"Total common (matched) tokens: {results_df['common_tokens'].sum()}")
print(f"Overall token coverage: {results_df['common_tokens'].sum() / results_df['expected_tokens'].sum() * 100:.2f}%")


SCORE DISTRIBUTION

Precision Statistics:
count    105.000000
mean       0.766655
std        0.159077
min        0.285714
25%        0.678571
50%        0.800000
75%        0.882353
max        1.000000
Name: precision, dtype: float64

Recall Statistics:
count    105.000000
mean       0.630517
std        0.185244
min        0.194444
25%        0.480000
50%        0.638298
75%        0.759259
max        1.000000
Name: recall, dtype: float64

F1 Score Statistics:
count    105.000000
mean       0.681689
std        0.158643
min        0.266667
25%        0.581818
50%        0.705882
75%        0.800000
max        1.000000
Name: f1, dtype: float64

SCORE RANGE DISTRIBUTION

F1 Score Distribution:
f1_category
Excellent (0.9-1.0)     6
Fair (0.5-0.7)         34
Good (0.7-0.9)         48
Poor (0.3-0.5)         14
Very Poor (0.0-0.3)     3
Name: count, dtype: int64

TOKEN COVERAGE ANALYSIS
Total expected tokens across all questions: 4036
Total predicted tokens across all questions: 3282
Total c

In [ ]:
# Show top 10 best performing questions
print("=" * 80)
print("TOP 10 BEST PERFORMING QUESTIONS (by F1 Score)")
print("=" * 80)

top_10 = results_df.nlargest(10, 'f1')[['question_id', 'precision', 'recall', 'f1', 'question']]
for idx, row in top_10.iterrows():
    print(f"\n{row['question_id']}. F1={row['f1']:.3f}, P={row['precision']:.3f}, R={row['recall']:.3f}")
    print(f"   Q: {row['question'][:100]}...")

# Show bottom 10 worst performing questions  
print("\n" + "=" * 80)
print("TOP 10 WORST PERFORMING QUESTIONS (by F1 Score)")
print("=" * 80)

bottom_10 = results_df.nsmallest(10, 'f1')[['question_id', 'precision', 'recall', 'f1', 'question']]
for idx, row in bottom_10.iterrows():
    print(f"\n{row['question_id']}. F1={row['f1']:.3f}, P={row['precision']:.3f}, R={row['recall']:.3f}")
    print(f"   Q: {row['question'][:100]}...")

TOP 10 BEST PERFORMING QUESTIONS (by F1 Score)

4. F1=1.000, P=1.000, R=1.000
   Q: What is the difference between soil health and soil quality?...

94. F1=0.960, P=0.923, R=1.000
   Q: How is land take defined, and what type of land is it usually at the expense of?...

10. F1=0.947, P=0.900, R=1.000
   Q: What are soil indicators and what is the relationship between soil indicators and soil attributes?...

5. F1=0.944, P=0.944, R=0.944
   Q: What are soil threats and what factors indicate the existence of soil threats?...

12. F1=0.914, P=1.000, R=0.842
   Q: What risk assessment models are currently in use and in which countries, if any, are they used?...

3. F1=0.902, P=0.920, R=0.885
   Q: What is the common definition of soil quality and soil health?...

7. F1=0.889, P=0.865, R=0.914
   Q: What are thresholds and what information do the thresholds provide in the context of soil protection...

20. F1=0.880, P=1.000, R=0.786
   Q: Which GAEC standards are relevant to soils, and what

In [ ]:
# Save detailed results to CSV
output_file = 'kg_validation_results.csv'
results_df.to_csv(output_file, index=False)
print(f"\n✓ Detailed evaluation results saved to: {output_file}")

# Also save a summary report
summary_file = 'kg_validation_summary.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("KNOWLEDGE GRAPH EVALUATION SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total Questions Evaluated: {all_summary['total_questions']}\n")
    f.write(f"Successful Evaluations: {all_summary['successful_evaluations']}\n")
    f.write(f"Failed Evaluations: {all_summary['failed_evaluations']}\n\n")
    f.write(f"Average Precision: {all_summary['average_precision']:.4f}\n")
    f.write(f"Average Recall: {all_summary['average_recall']:.4f}\n")
    f.write(f"Average F1 Score: {all_summary['average_f1']:.4f}\n\n")
    f.write(f"Questions with Perfect Recall (1.0): {all_summary['questions_with_perfect_recall']}\n")
    f.write(f"Questions with Perfect Precision (1.0): {all_summary['questions_with_perfect_precision']}\n")
    f.write(f"Questions with Perfect F1 (1.0): {all_summary['questions_with_perfect_f1']}\n\n")
    f.write("=" * 80 + "\n")
    f.write("SCORE DISTRIBUTION\n")
    f.write("=" * 80 + "\n\n")
    f.write(results_df['f1_category'].value_counts().sort_index().to_string())
    f.write("\n\n")
    f.write("=" * 80 + "\n")
    f.write("TOKEN COVERAGE\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total expected tokens: {results_df['expected_tokens'].sum()}\n")
    f.write(f"Total predicted tokens: {results_df['predicted_tokens'].sum()}\n")
    f.write(f"Total matched tokens: {results_df['common_tokens'].sum()}\n")
    f.write(f"Overall coverage: {results_df['common_tokens'].sum() / results_df['expected_tokens'].sum() * 100:.2f}%\n")

print(f"✓ Summary report saved to: {summary_file}")

print("\n" + "=" * 80)
print("EVALUATION COMPLETE!")
print("=" * 80)
print(f"\nThe knowledge graph achieves an average F1 score of {all_summary['average_f1']:.2%}")
print(f"across {all_summary['total_questions']} competency questions.")
print(f"\nThis indicates that the KG captures approximately {all_summary['average_recall']:.2%}")
print(f"of the expected information, with {all_summary['average_precision']:.2%} precision.")


✓ Detailed evaluation results saved to: kg_evaluation_results.csv
✓ Summary report saved to: kg_evaluation_summary.txt

EVALUATION COMPLETE!

The knowledge graph achieves an average F1 score of 68.17%
across 105 competency questions.

This indicates that the KG captures approximately 63.05%
of the expected information, with 76.67% precision.


#### LLM-Based Answer Rewriting

In [27]:
# System prompt for LLM rewriting
SYSTEM_PROMPT = """## KG Rewriting

You are a "Knowledge Graph-to-Text" summarization bot. Your sole function is to receive raw, factual output from a knowledge graph (KG) query and rewrite it into a single, coherent, human-readable answer.

You will be given the original **User Question** (for context) and the **KG Output** (the facts to use).

**CRITICAL RULES:**

1. **STRICTLY CONCISE:** Your rewritten answer must be as concise as possible. Synthesize and aggregate related facts into a single sentence. For example, instead of "A is X. B is X. C is X," write "A, B, and C are X." Do not use filler words or conversational phrases.
2. **NO EXTERNAL INFORMATION:** You MUST NOT add *any* information, explanation, or context that is not explicitly present in the **KG Output**. Your knowledge is strictly limited to the provided facts. Do not elaborate, define terms, or provide background unless it is in the KG output.

**Example Task:**

**User Question:** "What laptop brands are manufactured by TechGlobal for the European market, and what does a standard warranty cover?"

**KG Output:** "AetherBook is brand of TechGlobal. AetherBook is sold in Europe. NovaPad is brand of TechGlobal. NovaPad is sold in Europe. Zenion is brand of TechGlobal. Zenion is sold in Europe. standard warranty includes Parts. standard warranty includes Labor. standard warranty includes Support."

**Your Rewritten Answer:** "AetherBook, NovaPad, and Zenion are laptop brands manufactured by TechGlobal for the European market. A standard warranty covers parts, labor, and support."
"""

In [28]:
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def rewrite_answer_with_llm(question, kg_output, client, model="gpt-6-luna", max_retries=3):
    """
    Use LLM to rewrite KG output into a coherent human-readable answer.
    
    Parameters:
    - question: The original competency question
    - kg_output: The raw KG query output (predicted_answer)
    - client: OpenAI client instance
    - model: Model to use (default: gpt-6-luna)
    - max_retries: Number of retries on failure
    
    Returns:
    - Rewritten answer string, or original kg_output if rewriting fails
    """
    if not client:
        return kg_output  # Return original if no client
    
    # Create the user prompt
    user_prompt = f"""**User Question:** "{question}"

**KG Output:** "{kg_output}"

**Your Rewritten Answer:**"""
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ]
            )
            
            rewritten_answer = response.choices[0].message.content.strip()
            return rewritten_answer
            
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} due to error: {e}")
                import time
                time.sleep(2)  # Wait before retrying
            else:
                print(f"  ❌ Failed after {max_retries} attempts: {e}")
                return kg_output  # Return original on failure
    
    return kg_output

print("✓ LLM rewriting function defined successfully!")

✓ LLM rewriting function defined successfully!


In [ ]:
def rewrite_all_answers(csv_file_path, client, model="gpt-6-luna", save_as=None, max_questions=None):
    """
    Rewrite all predicted answers in the CSV file using LLM.
    
    Parameters:
    - csv_file_path: Path to the kg_validation_results.csv file
    - client: OpenAI client instance
    - model: Model to use
    - save_as: Filename to save rewritten results (None = overwrite original)
    - max_questions: Maximum questions to process (None = all)
    
    Returns:
    - DataFrame with rewritten answers
    """
    if not client:
        print("❌ OpenAI client not initialized. Cannot rewrite answers.")
        return None
    
    # Load the results
    df = pd.read_csv(csv_file_path)
    
    if max_questions:
        df = df.head(max_questions)
    
    print(f"Rewriting {len(df)} answers using {model}...")
    print("=" * 80)
    
    # Add a new column for rewritten answers
    df['llm_rewritten_answer'] = ''
    
    for idx, row in df.iterrows():
        question = row['question']
        kg_output = row['predicted_answer']
        question_id = row['question_id']
        
        print(f"Processing Question {question_id}/{len(df)}...", end=' ')
        
        # Rewrite the answer
        rewritten = rewrite_answer_with_llm(question, kg_output, client, model)
        df.at[idx, 'llm_rewritten_answer'] = rewritten
        
        print("✓")
    
    print("\n" + "=" * 80)
    print(f"✓ Completed rewriting {len(df)} answers!")
    
    # Save the results
    output_file = save_as if save_as else csv_file_path.replace('.csv', '_llm_rewritten.csv')
    df.to_csv(output_file, index=False)
    print(f"✓ Saved to: {output_file}")
    
    return df

print("✓ Batch rewriting function defined successfully!")

✓ Batch rewriting function defined successfully!


In [ ]:
# Rewrite ALL answers (uncomment to run)
if client:
    print("=" * 80)
    print("REWRITING ALL 105 ANSWERS WITH LLM")
    print("=" * 80)
    print("⚠️  This will make 105 API calls to OpenAI.")
    print("⚠️  Estimated cost: ~$0.50-$2.00 depending on token usage")
    print("")
    
    # Uncomment the line below to proceed with rewriting all answers
    all_rewritten_df = rewrite_all_answers('kg_validation_results.csv', client)
    
    # For now, just show instructions
    print("To proceed:")
    print("1. Uncomment the line: all_rewritten_df = rewrite_all_answers(...)")
    print("2. Run this cell")
    print("\nThe results will be saved to: kg_validation_results_llm_rewritten.csv")
else:
    print("⚠️ OpenAI client not initialized. Cannot proceed with rewriting.")

REWRITING ALL 105 ANSWERS WITH LLM
⚠️  This will make 105 API calls to OpenAI.
⚠️  Estimated cost: ~$0.50-$2.00 depending on token usage

Rewriting 105 answers using gpt-5...
Processing Question 1/105... ✓
Processing Question 2/105... ✓
Processing Question 3/105... ✓
Processing Question 4/105... ✓
Processing Question 5/105... ✓
Processing Question 6/105... ✓
Processing Question 7/105... ✓
Processing Question 8/105... ✓
Processing Question 9/105... ✓
Processing Question 10/105... ✓
Processing Question 11/105... ✓
Processing Question 12/105... ✓
Processing Question 13/105... ✓
Processing Question 14/105... ✓
Processing Question 15/105... ✓
Processing Question 16/105... ✓
Processing Question 17/105... ✓
Processing Question 18/105... ✓
Processing Question 19/105... ✓
Processing Question 20/105... ✓
Processing Question 21/105... ✓
Processing Question 22/105... ✓
Processing Question 23/105... ✓
Processing Question 24/105... ✓
Processing Question 25/105... ✓
Processing Question 26/105... ✓
Pr

In [ ]:
def reevaluate_with_llm_answers(rewritten_csv_path):
    """
    Re-calculate metrics using LLM-rewritten answers instead of original predicted answers.
    
    Parameters:
    - rewritten_csv_path: Path to CSV with llm_rewritten_answer column
    
    Returns:
    - DataFrame with new metrics and comparison
    """
    # Load the rewritten results
    df = pd.read_csv(rewritten_csv_path)
    
    if 'llm_rewritten_answer' not in df.columns:
        print("❌ Error: CSV file does not contain 'llm_rewritten_answer' column")
        return None
    
    print(f"Re-evaluating {len(df)} questions with LLM-rewritten answers...")
    print("=" * 80)
    
    # Calculate new metrics for LLM-rewritten answers
    llm_metrics = []
    
    for idx, row in df.iterrows():
        expected = row['expected_answer']
        llm_answer = row['llm_rewritten_answer']
        
        # Calculate metrics
        metrics = calculate_token_metrics(llm_answer, expected)
        llm_metrics.append(metrics)
    
    # Add new metrics columns
    df['llm_precision'] = [m['precision'] for m in llm_metrics]
    df['llm_recall'] = [m['recall'] for m in llm_metrics]
    df['llm_f1'] = [m['f1'] for m in llm_metrics]
    df['llm_common_tokens'] = [m['common_tokens'] for m in llm_metrics]
    df['llm_predicted_tokens'] = [m['predicted_tokens'] for m in llm_metrics]
    
    # Calculate improvement
    df['f1_improvement'] = df['llm_f1'] - df['f1']
    df['precision_improvement'] = df['llm_precision'] - df['precision']
    df['recall_improvement'] = df['llm_recall'] - df['recall']
    
    print("✓ Re-evaluation complete!")
    
    # Calculate summary statistics
    original_avg_f1 = df['f1'].mean()
    llm_avg_f1 = df['llm_f1'].mean()
    original_avg_precision = df['precision'].mean()
    llm_avg_precision = df['llm_precision'].mean()
    original_avg_recall = df['recall'].mean()
    llm_avg_recall = df['llm_recall'].mean()
    
    summary = {
        'total_questions': len(df),
        'original_avg_precision': original_avg_precision,
        'original_avg_recall': original_avg_recall,
        'original_avg_f1': original_avg_f1,
        'llm_avg_precision': llm_avg_precision,
        'llm_avg_recall': llm_avg_recall,
        'llm_avg_f1': llm_avg_f1,
        'avg_f1_improvement': (llm_avg_f1 - original_avg_f1),
        'avg_precision_improvement': (llm_avg_precision - original_avg_precision),
        'avg_recall_improvement': (llm_avg_recall - original_avg_recall),
        'questions_improved': len(df[df['f1_improvement'] > 0]),
        'questions_degraded': len(df[df['f1_improvement'] < 0]),
        'questions_unchanged': len(df[df['f1_improvement'] == 0])
    }
    
    return df, summary

print("✓ Re-evaluation function defined successfully!")

✓ Re-evaluation function defined successfully!


In [ ]:
# Run re-evaluation (uncomment after rewriting all answers)
# This will compare original vs LLM-rewritten performance

# Uncomment the lines below after running the full rewriting:

comparison_df, comparison_summary = reevaluate_with_llm_answers(
    'kg_validation/kg_validation_results_llm_rewritten.csv'
)

if comparison_df is not None:
    print("\n" + "=" * 80)
    print("COMPARISON: ORIGINAL vs LLM-REWRITTEN PERFORMANCE")
    print("=" * 80)
    
    print(f"\n{'Metric':<25} {'Original':<12} {'LLM-Rewritten':<15} {'Improvement'}")
    print("-" * 80)
    print(f"{'Average Precision:':<25} {comparison_summary['original_avg_precision']:<12.4f} {comparison_summary['llm_avg_precision']:<15.4f} {comparison_summary['avg_precision_improvement']:+.4f}")
    print(f"{'Average Recall:':<25} {comparison_summary['original_avg_recall']:<12.4f} {comparison_summary['llm_avg_recall']:<15.4f} {comparison_summary['avg_recall_improvement']:+.4f}")
    print(f"{'Average F1 Score:':<25} {comparison_summary['original_avg_f1']:<12.4f} {comparison_summary['llm_avg_f1']:<15.4f} {comparison_summary['avg_f1_improvement']:+.4f}")
    
    print(f"\n{'Performance Changes':<40} {'Count':<10} {'Percentage'}")
    print("-" * 80)
    total = comparison_summary['total_questions']
    print(f"{'Questions with Improved F1:':<40} {comparison_summary['questions_improved']:<10} {comparison_summary['questions_improved']/total*100:.1f}%")
    print(f"{'Questions with Degraded F1:':<40} {comparison_summary['questions_degraded']:<10} {comparison_summary['questions_degraded']/total*100:.1f}%")
    print(f"{'Questions with Unchanged F1:':<40} {comparison_summary['questions_unchanged']:<10} {comparison_summary['questions_unchanged']/total*100:.1f}%")
    
    # Save the comparison results
    comparison_df.to_csv('kg_validation_comparison.csv', index=False)
    print(f"\n✓ Detailed comparison saved to: kg_validation_comparison.csv")


print("Uncomment the code above to run re-evaluation after rewriting all answers.")

Re-evaluating 105 questions with LLM-rewritten answers...
✓ Re-evaluation complete!

COMPARISON: ORIGINAL vs LLM-REWRITTEN PERFORMANCE

Metric                    Original     LLM-Rewritten   Improvement
--------------------------------------------------------------------------------
Average Precision:        0.7667       0.7871          +0.0204
Average Recall:           0.6305       0.6520          +0.0215
Average F1 Score:         0.6817       0.7062          +0.0245

Performance Changes                      Count      Percentage
--------------------------------------------------------------------------------
Questions with Improved F1:              63         60.0%
Questions with Degraded F1:              40         38.1%
Questions with Unchanged F1:             2          1.9%

✓ Detailed comparison saved to: kg_evaluation_comparison.csv
Uncomment the code above to run re-evaluation after rewriting all answers.


In [ ]:
# Show examples of biggest improvements (uncomment after re-evaluation)

if comparison_df is not None:
    print("\n" + "=" * 80)
    print("TOP 5 BIGGEST IMPROVEMENTS")
    print("=" * 80)
    
    top_improvements = comparison_df.nlargest(5, 'f1_improvement')
    
    for idx, row in top_improvements.iterrows():
        print(f"\nQuestion {row['question_id']}: {row['question'][:80]}...")
        print(f"F1 Score: {row['f1']:.3f} → {row['llm_f1']:.3f} (Δ {row['f1_improvement']:+.3f})")
        print(f"\nExpected: {row['expected_answer'][:100]}...")
        print(f"\nOriginal KG: {row['predicted_answer'][:100]}...")
        print(f"\nLLM Rewritten: {row['llm_rewritten_answer'][:150]}...")
        print("-" * 80)
    
    print("\n" + "=" * 80)
    print("TOP 5 BIGGEST DEGRADATIONS")
    print("=" * 80)
    
    top_degradations = comparison_df.nsmallest(5, 'f1_improvement')
    
    for idx, row in top_degradations.iterrows():
        print(f"\nQuestion {row['question_id']}: {row['question'][:80]}...")
        print(f"F1 Score: {row['f1']:.3f} → {row['llm_f1']:.3f} (Δ {row['f1_improvement']:+.3f})")
        print("-" * 80)


print("Uncomment to show improvement examples after re-evaluation.")


TOP 5 BIGGEST IMPROVEMENTS

Question 13: What are the main types of soil degradation and where does soil sealing fall int...
F1 Score: 0.583 → 1.000 (Δ +0.417)

Expected: The main types of soil degradation are physical, chemical, biological, and ecological degradation. S...

Original KG: soil degradation includes soil biological degradation. soil degradation includes soil chemical degra...

LLM Rewritten: The main types of soil degradation are biological, chemical, ecological, and physical, and soil sealing is a form of physical degradation....
--------------------------------------------------------------------------------

Question 69: What are the specific soil biodiversity indicators proposed by Huber et al. (200...
F1 Score: 0.436 → 0.741 (Δ +0.304)

Expected: Huber et al. (2008) proposed three key indicators: (1) diversity of earthworms; (2) diversity of col...

Original KG: bacterial biomass source Breure2004. diversity of collembolans source Huber2008. diversity of earthw...



#### BERTScore Evaluation

In [ ]:
def calculate_bertscore(predicted_answer, expected_answer, model_type="microsoft/deberta-xlarge-mnli"):
    """
    Calculate BERTScore metrics for a single Q&A pair.
    
    Args:
        predicted_answer: The predicted answer from the KG
        expected_answer: The expected (ground truth) answer
        model_type: The pre-trained model to use for embeddings
                   Options: "microsoft/deberta-xlarge-mnli" (best but slower)
                           "roberta-large" (good balance)
                           "bert-base-uncased" (fastest but less accurate)
    
    Returns:
        Dictionary with precision, recall, and F1 scores
    """
    # Handle empty strings
    if not predicted_answer or not predicted_answer.strip():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    if not expected_answer or not expected_answer.strip():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    
    # BERTScore expects lists
    P, R, F1 = score(
        [predicted_answer], 
        [expected_answer], 
        model_type=model_type,
        verbose=False
    )
    
    return {
        "precision": P.item(),
        "recall": R.item(), 
        "f1": F1.item()
    }


In [ ]:
def evaluate_with_bertscore(results_df, model_type="microsoft/deberta-xlarge-mnli"):
    """
    Evaluate all Q&A pairs using BERTScore and add scores to the dataframe.
    
    Args:
        results_df: DataFrame with 'expected_answer' and 'predicted_answer' columns
        model_type: The pre-trained model to use for BERTScore
    
    Returns:
        DataFrame with added BERTScore columns: bert_precision, bert_recall, bert_f1
    """
    print(f"Evaluating {len(results_df)} Q&A pairs with BERTScore...")
    print(f"Using model: {model_type}")
    print("This may take a few minutes...")
    
    # Extract lists of predicted and expected answers
    predicted_answers = results_df['llm_rewritten_answer'].tolist()
    expected_answers = results_df['expected_answer'].tolist()
    
    # Calculate BERTScore for all pairs at once (more efficient)
    P, R, F1 = score(
        predicted_answers,
        expected_answers,
        model_type=model_type,
        verbose=True
    )
    
    # Add scores to dataframe
    results_df['bert_precision'] = P.cpu().numpy()
    results_df['bert_recall'] = R.cpu().numpy()
    results_df['bert_f1'] = F1.cpu().numpy()
    
    # Calculate summary statistics
    print("\n" + "="*60)
    print("BERTScore Evaluation Results:")
    print("="*60)
    print(f"Average BERTScore Precision: {results_df['bert_precision'].mean():.4f}")
    print(f"Average BERTScore Recall:    {results_df['bert_recall'].mean():.4f}")
    print(f"Average BERTScore F1:        {results_df['bert_f1'].mean():.4f}")
    print("="*60)
    
    return results_df

In [ ]:
# Load the existing evaluation results
results_with_bert = pd.read_csv('kg_validation_comparison.csv')

# Add BERTScore evaluation
# Using bert-base-uncased for faster evaluation (you can change to microsoft/deberta-xlarge-mnli for better accuracy)
results_with_bert = evaluate_with_bertscore(results_with_bert)

# Save updated results
results_with_bert.to_csv('kg_validation_with_bertscore.csv', index=False)
print("\nResults saved to 'kg_validation_with_bertscore.csv'")

Evaluating 105 Q&A pairs with BERTScore...
Using model: microsoft/deberta-xlarge-mnli
This may take a few minutes...
calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 483.89 seconds, 0.22 sentences/sec

BERTScore Evaluation Results:
Average BERTScore Precision: 0.8088
Average BERTScore Recall:    0.7779
Average BERTScore F1:        0.7923

Results saved to 'kg_evaluation_with_bertscore.csv'


In [ ]:
# Compare Token-based F1 vs BERTScore F1
print("="*80)
print("COMPARISON: Token-based F1 vs BERTScore F1")
print("="*80)

comparison_stats = {
    'Token-based F1': {
        'Mean': results_with_bert['f1'].mean(),
        'Median': results_with_bert['f1'].median(),
        'Std': results_with_bert['f1'].std(),
        'Min': results_with_bert['f1'].min(),
        'Max': results_with_bert['f1'].max()
    },
    'BERTScore F1': {
        'Mean': results_with_bert['bert_f1'].mean(),
        'Median': results_with_bert['bert_f1'].median(),
        'Std': results_with_bert['bert_f1'].std(),
        'Min': results_with_bert['bert_f1'].min(),
        'Max': results_with_bert['bert_f1'].max()
    }
}

for metric_name, stats in comparison_stats.items():
    print(f"\n{metric_name}:")
    print(f"  Mean:   {stats['Mean']:.4f}")
    print(f"  Median: {stats['Median']:.4f}")
    print(f"  Std:    {stats['Std']:.4f}")
    print(f"  Range:  [{stats['Min']:.4f}, {stats['Max']:.4f}]")

# Calculate correlation
correlation = results_with_bert['f1'].corr(results_with_bert['bert_f1'])
print(f"\nCorrelation between Token F1 and BERTScore F1: {correlation:.4f}")

# Show improvement
results_with_bert['f1_improvement'] = results_with_bert['bert_f1'] - results_with_bert['f1']
avg_improvement = results_with_bert['f1_improvement'].mean()
print(f"\nAverage improvement (BERTScore F1 - Token F1): {avg_improvement:.4f}")
print(f"Questions with improved scores: {(results_with_bert['f1_improvement'] > 0).sum()} / {len(results_with_bert)}")

COMPARISON: Token-based F1 vs BERTScore F1

Token-based F1:
  Mean:   0.6817
  Median: 0.7059
  Std:    0.1586
  Range:  [0.2667, 1.0000]

BERTScore F1:
  Mean:   0.7923
  Median: 0.7858
  Std:    0.0714
  Range:  [0.6185, 0.9514]

Correlation between Token F1 and BERTScore F1: 0.6331

Average improvement (BERTScore F1 - Token F1): 0.1106
Questions with improved scores: 87 / 105


In [ ]:
# Show examples where BERTScore significantly differs from Token F1
print("="*80)
print("EXAMPLES: Cases where BERTScore captures semantic similarity better")
print("="*80)

# Find cases with biggest positive improvements (BERTScore > Token F1)
top_improvements = results_with_bert.nlargest(5, 'f1_improvement')[
    ['question', 'expected_answer', 'predicted_answer', 'f1', 'bert_f1', 'f1_improvement']
]

print("\nTop 5 cases where BERTScore found better semantic match:")
print("-" * 80)
for idx, row in top_improvements.iterrows():
    print(f"\nQuestion: {row['question'][:100]}...")
    print(f"Expected: {row['expected_answer'][:150]}...")
    print(f"Predicted: {row['predicted_answer'][:150]}...")
    print(f"Token F1: {row['f1']:.4f} → BERTScore F1: {row['bert_f1']:.4f} (Δ = +{row['f1_improvement']:.4f})")
    print("-" * 80)

# Find cases with biggest negative differences (Token F1 > BERTScore)
top_degradations = results_with_bert.nsmallest(5, 'f1_improvement')[
    ['question', 'expected_answer', 'predicted_answer', 'f1', 'bert_f1', 'f1_improvement']
]

print("\n\nTop 5 cases where Token F1 was higher:")
print("-" * 80)
for idx, row in top_degradations.iterrows():
    print(f"\nQuestion: {row['question'][:100]}...")
    print(f"Expected: {row['expected_answer'][:150]}...")
    print(f"Predicted: {row['predicted_answer'][:150]}...")
    print(f"Token F1: {row['f1']:.4f} → BERTScore F1: {row['bert_f1']:.4f} (Δ = {row['f1_improvement']:.4f})")
    print("-" * 80)

EXAMPLES: Cases where BERTScore captures semantic similarity better

Top 5 cases where BERTScore found better semantic match:
--------------------------------------------------------------------------------

Question: What is the soil pH range that can constrain crop production and what are the critical and optimal p...
Expected: Crop production is already constrained at pH values below 5.5-6.0. A pH value below 5 should certainly be avoided. Ideally, the pH should stay above 5...
Predicted: soil pH greater than 5.5 beneficial for crop production. soil pH greater than 6.0 beneficial for crop production. soil pH smaller than 5.5-6.0 constra...
Token F1: 0.2667 → BERTScore F1: 0.6896 (Δ = +0.4229)
--------------------------------------------------------------------------------

Question: What key elements need to be considered when establishing meaningful thresholds for pollutants in so...
Expected: To derive meaningful thresholds for pollutants in soil, the entire pathway — from emissio

In [ ]:
# Performance categorization with BERTScore
print("="*80)
print("PERFORMANCE BREAKDOWN BY BERTSCORE F1")
print("="*80)

def categorize_bertscore(score):
    if score >= 0.9:
        return "Excellent (≥0.9)"
    elif score >= 0.8:
        return "Good (0.8-0.9)"
    elif score >= 0.7:
        return "Fair (0.7-0.8)"
    elif score >= 0.5:
        return "Poor (0.5-0.7)"
    else:
        return "Very Poor (<0.5)"

results_with_bert['bert_category'] = results_with_bert['bert_f1'].apply(categorize_bertscore)
bert_category_counts = results_with_bert['bert_category'].value_counts()

print("\nDistribution by BERTScore F1 category:")
for category in ["Excellent (≥0.9)", "Good (0.8-0.9)", "Fair (0.7-0.8)", "Poor (0.5-0.7)", "Very Poor (<0.5)"]:
    count = bert_category_counts.get(category, 0)
    percentage = (count / len(results_with_bert)) * 100
    print(f"{category:20s}: {count:3d} questions ({percentage:5.1f}%)")

# Compare with token-based categories
results_with_bert['token_category'] = results_with_bert['f1'].apply(categorize_bertscore)
token_category_counts = results_with_bert['token_category'].value_counts()

print("\n" + "="*80)
print("COMPARISON: Category Distribution")
print("="*80)
print(f"{'Category':<20s} {'Token F1':>15s} {'BERTScore F1':>15s} {'Difference':>12s}")
print("-" * 80)

for category in ["Excellent (≥0.9)", "Good (0.8-0.9)", "Fair (0.7-0.8)", "Poor (0.5-0.7)", "Very Poor (<0.5)"]:
    token_count = token_category_counts.get(category, 0)
    bert_count = bert_category_counts.get(category, 0)
    diff = bert_count - token_count
    diff_str = f"{diff:+d}" if diff != 0 else "0"
    print(f"{category:<20s} {token_count:>15d} {bert_count:>15d} {diff_str:>12s}")

print("="*80)

PERFORMANCE BREAKDOWN BY BERTSCORE F1

Distribution by BERTScore F1 category:
Excellent (≥0.9)    :   7 questions (  6.7%)
Good (0.8-0.9)      :  38 questions ( 36.2%)
Fair (0.7-0.8)      :  50 questions ( 47.6%)
Poor (0.5-0.7)      :  10 questions (  9.5%)
Very Poor (<0.5)    :   0 questions (  0.0%)

COMPARISON: Category Distribution
Category                    Token F1    BERTScore F1   Difference
--------------------------------------------------------------------------------
Excellent (≥0.9)                   6               7           +1
Good (0.8-0.9)                    20              38          +18
Fair (0.7-0.8)                    28              50          +22
Poor (0.5-0.7)                    34              10          -24
Very Poor (<0.5)                  17               0          -17


#### Embedding-Based Similarity Evaluation

In [ ]:
def get_openai_embedding(text, model="text-embedding-3-small"):
    """
    Get embedding vector for a text using OpenAI's embedding API.
    
    Args:
        text: The text to embed
        model: OpenAI embedding model to use
               - "text-embedding-3-small": Fast, cost-effective (1536 dimensions)
               - "text-embedding-3-large": Higher quality (3072 dimensions)
    
    Returns:
        numpy array of embedding vector
    """
    # Handle empty text
    if not text or not text.strip():
        # Return zero vector for empty text
        dim = 1536 if model == "text-embedding-3-small" else 3072
        return np.zeros(dim)
    
    # Replace newlines for better API handling
    text = text.replace("\n", " ")
    
    # Get embedding from OpenAI API
    response = client.embeddings.create(
        input=[text],
        model=model
    )
    
    # Extract embedding vector
    embedding = np.array(response.data[0].embedding)
    return embedding


def calculate_embedding_similarity(predicted_answer, expected_answer, model="text-embedding-3-small"):
    """
    Calculate cosine similarity between two texts using OpenAI embeddings.
    
    Args:
        predicted_answer: The predicted answer from KG
        expected_answer: The expected (ground truth) answer
        model: OpenAI embedding model to use
    
    Returns:
        Float between 0 and 1 representing cosine similarity
    """
    # Get embeddings for both texts
    pred_embedding = get_openai_embedding(predicted_answer, model)
    exp_embedding = get_openai_embedding(expected_answer, model)
    
    # Calculate cosine similarity
    # Reshape for sklearn's cosine_similarity function
    similarity = cosine_similarity(
        pred_embedding.reshape(1, -1),
        exp_embedding.reshape(1, -1)
    )[0][0]
    
    return similarity

In [ ]:
def evaluate_with_openai_embeddings(results_df, model="text-embedding-3-small", batch_delay=0.1):
    """
    Evaluate all Q&A pairs using OpenAI embedding similarity.
    
    Args:
        results_df: DataFrame with 'expected_answer' and 'predicted_answer' columns
        model: OpenAI embedding model to use
        batch_delay: Delay between API calls to avoid rate limits (seconds)
    
    Returns:
        DataFrame with added 'embedding_similarity' column
    """
    print(f"Evaluating {len(results_df)} Q&A pairs with OpenAI Embeddings...")
    print(f"Using model: {model}")
    print("This may take a few minutes...")
    
    similarities = []
    
    for idx, row in results_df.iterrows():
        try:
            similarity = calculate_embedding_similarity(
                row['llm_rewritten_answer'],
                row['expected_answer'],
                model=model
            )
            similarities.append(similarity)
            
            # Progress indicator
            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(results_df)} questions...")
            
            # Small delay to respect rate limits
            time.sleep(batch_delay)
            
        except Exception as e:
            print(f"Error processing question {idx}: {e}")
            similarities.append(0.0)  # Default to 0 on error
    
    # Add similarities to dataframe
    results_df['embedding_similarity'] = similarities
    
    # Calculate summary statistics
    print("\n" + "="*60)
    print("OpenAI Embedding Similarity Results:")
    print("="*60)
    print(f"Average Similarity: {results_df['embedding_similarity'].mean():.4f}")
    print(f"Median Similarity:  {results_df['embedding_similarity'].median():.4f}")
    print(f"Std Dev:            {results_df['embedding_similarity'].std():.4f}")
    print(f"Min Similarity:     {results_df['embedding_similarity'].min():.4f}")
    print(f"Max Similarity:     {results_df['embedding_similarity'].max():.4f}")
    print("="*60)
    
    return results_df

In [ ]:
# Load the existing evaluation results with BERTScore
results_with_embeddings = pd.read_csv('kg_validation_with_bertscore.csv')

# Add OpenAI embedding similarity evaluation
# Using text-embedding-3-small for cost-effectiveness (you can change to text-embedding-3-large for better quality)
results_with_embeddings = evaluate_with_openai_embeddings(
    results_with_embeddings, 
    model="text-embedding-3-large",
    batch_delay=0.1  # 0.1 second delay between calls
)

# Save updated results
results_with_embeddings.to_csv('kg_validation_complete.csv', index=False)
print("\nResults saved to 'kg_validation_complete.csv'")

Evaluating 105 Q&A pairs with OpenAI Embeddings...
Using model: text-embedding-3-large
This may take a few minutes...
Processed 10/105 questions...
Processed 20/105 questions...
Processed 30/105 questions...
Processed 40/105 questions...
Processed 50/105 questions...
Processed 60/105 questions...
Processed 70/105 questions...
Processed 80/105 questions...
Processed 90/105 questions...
Processed 100/105 questions...

OpenAI Embedding Similarity Results:
Average Similarity: 0.8945
Median Similarity:  0.9123
Std Dev:            0.0608
Min Similarity:     0.7125
Max Similarity:     0.9887

Results saved to 'kg_evaluation_complete.csv'


In [ ]:
# Compare all three metrics: Token F1, BERTScore F1, and OpenAI Embedding Similarity
print("="*80)
print("COMPREHENSIVE COMPARISON: Three Evaluation Metrics")
print("="*80)

comparison_stats = {
    'Token-based F1': {
        'Mean': results_with_embeddings['f1'].mean(),
        'Median': results_with_embeddings['f1'].median(),
        'Std': results_with_embeddings['f1'].std(),
        'Min': results_with_embeddings['f1'].min(),
        'Max': results_with_embeddings['f1'].max()
    },
    'BERTScore F1': {
        'Mean': results_with_embeddings['bert_f1'].mean(),
        'Median': results_with_embeddings['bert_f1'].median(),
        'Std': results_with_embeddings['bert_f1'].std(),
        'Min': results_with_embeddings['bert_f1'].min(),
        'Max': results_with_embeddings['bert_f1'].max()
    },
    'OpenAI Embedding Similarity': {
        'Mean': results_with_embeddings['embedding_similarity'].mean(),
        'Median': results_with_embeddings['embedding_similarity'].median(),
        'Std': results_with_embeddings['embedding_similarity'].std(),
        'Min': results_with_embeddings['embedding_similarity'].min(),
        'Max': results_with_embeddings['embedding_similarity'].max()
    }
}

for metric_name, stats in comparison_stats.items():
    print(f"\n{metric_name}:")
    print(f"  Mean:   {stats['Mean']:.4f}")
    print(f"  Median: {stats['Median']:.4f}")
    print(f"  Std:    {stats['Std']:.4f}")
    print(f"  Range:  [{stats['Min']:.4f}, {stats['Max']:.4f}]")

# Calculate correlations between metrics
print("\n" + "="*80)
print("CORRELATIONS BETWEEN METRICS:")
print("="*80)
corr_token_bert = results_with_embeddings['f1'].corr(results_with_embeddings['bert_f1'])
corr_token_embed = results_with_embeddings['f1'].corr(results_with_embeddings['embedding_similarity'])
corr_bert_embed = results_with_embeddings['bert_f1'].corr(results_with_embeddings['embedding_similarity'])

print(f"Token F1 vs BERTScore F1:           {corr_token_bert:.4f}")
print(f"Token F1 vs Embedding Similarity:   {corr_token_embed:.4f}")
print(f"BERTScore F1 vs Embedding Similarity: {corr_bert_embed:.4f}")
print("="*80)

COMPREHENSIVE COMPARISON: Three Evaluation Metrics

Token-based F1:
  Mean:   0.6817
  Median: 0.7059
  Std:    0.1586
  Range:  [0.2667, 1.0000]

BERTScore F1:
  Mean:   0.7923
  Median: 0.7858
  Std:    0.0714
  Range:  [0.6185, 0.9514]

OpenAI Embedding Similarity:
  Mean:   0.8945
  Median: 0.9123
  Std:    0.0608
  Range:  [0.7125, 0.9887]

CORRELATIONS BETWEEN METRICS:
Token F1 vs BERTScore F1:           0.6331
Token F1 vs Embedding Similarity:   0.4718
BERTScore F1 vs Embedding Similarity: 0.6135


In [ ]:
# Show examples where different metrics agree or disagree
print("="*80)
print("EXAMPLES: Where Metrics Agree and Disagree")
print("="*80)

# Find questions where all metrics are high (all > 0.8)
all_high = results_with_embeddings[
    (results_with_embeddings['f1'] > 0.8) &
    (results_with_embeddings['bert_f1'] > 0.8) &
    (results_with_embeddings['embedding_similarity'] > 0.8)
]

print(f"\n✓ Questions where ALL metrics agree (all > 0.8): {len(all_high)}")
if len(all_high) > 0:
    print("\nTop example:")
    example = all_high.iloc[0]
    print(f"Question: {example['question'][:100]}...")
    print(f"Expected: {example['expected_answer'][:100]}...")
    print(f"Predicted: {example['predicted_answer'][:100]}...")
    print(f"  Token F1: {example['f1']:.4f}")
    print(f"  BERTScore F1: {example['bert_f1']:.4f}")
    print(f"  Embedding Sim: {example['embedding_similarity']:.4f}")

# Find questions where OpenAI Embedding is much higher than others
results_with_embeddings['embed_advantage'] = (
    results_with_embeddings['embedding_similarity'] - 
    results_with_embeddings[['f1', 'bert_f1']].mean(axis=1)
)

high_embed = results_with_embeddings.nlargest(3, 'embed_advantage')[
    ['question', 'expected_answer', 'predicted_answer', 'f1', 'bert_f1', 'embedding_similarity', 'embed_advantage']
]

print("\n" + "-"*80)
print("✓ Top 3 cases where Embedding Similarity outperforms other metrics:")
print("-"*80)
for idx, row in high_embed.iterrows():
    print(f"\nQuestion: {row['question'][:100]}...")
    print(f"Expected: {row['expected_answer'][:80]}...")
    print(f"Predicted: {row['predicted_answer'][:80]}...")
    print(f"  Token F1: {row['f1']:.4f} | BERTScore F1: {row['bert_f1']:.4f} | Embedding Sim: {row['embedding_similarity']:.4f}")
    print(f"  Advantage: +{row['embed_advantage']:.4f}")

# Find questions where traditional metrics are higher
results_with_embeddings['traditional_advantage'] = (
    results_with_embeddings[['f1', 'bert_f1']].mean(axis=1) -
    results_with_embeddings['embedding_similarity']
)

high_traditional = results_with_embeddings.nlargest(3, 'traditional_advantage')[
    ['question', 'expected_answer', 'predicted_answer', 'f1', 'bert_f1', 'embedding_similarity', 'traditional_advantage']
]

print("\n" + "-"*80)
print("✓ Top 3 cases where Token F1/BERTScore outperform Embedding Similarity:")
print("-"*80)
for idx, row in high_traditional.iterrows():
    print(f"\nQuestion: {row['question'][:100]}...")
    print(f"Expected: {row['expected_answer'][:80]}...")
    print(f"Predicted: {row['predicted_answer'][:80]}...")
    print(f"  Token F1: {row['f1']:.4f} | BERTScore F1: {row['bert_f1']:.4f} | Embedding Sim: {row['embedding_similarity']:.4f}")
    print(f"  Disadvantage: -{row['traditional_advantage']:.4f}")

print("="*80)

EXAMPLES: Where Metrics Agree and Disagree

✓ Questions where ALL metrics agree (all > 0.8): 18

Top example:
Question: What are the components of soil, and what are the relationships between soil and its functions?...
Expected: Soil is composed of a mixture of mineral and organic compounds, water, air and living organisms, and...
Predicted: soils is composed of air. soils is composed of living organisms. soils is composed of mineral compou...
  Token F1: 0.8125
  BERTScore F1: 0.9167
  Embedding Sim: 0.9235

--------------------------------------------------------------------------------
✓ Top 3 cases where Embedding Similarity outperforms other metrics:
--------------------------------------------------------------------------------

Question: What are the range of natural soil formation rates reported from several studies and what range of t...
Expected: Soil formation rates found in the literature vary quite significantly, as early ...
Predicted: soil formation rates 3.2-4.5 TONNE-

In [ ]:
# Create a final summary table for easy reference
summary_table = pd.DataFrame({
    'Metric': ['Token F1', 'BERTScore F1', 'OpenAI Embedding Sim'],
    'Mean': [
        results_with_embeddings['f1'].mean(),
        results_with_embeddings['bert_f1'].mean(),
        results_with_embeddings['embedding_similarity'].mean()
    ],
    'Median': [
        results_with_embeddings['f1'].median(),
        results_with_embeddings['bert_f1'].median(),
        results_with_embeddings['embedding_similarity'].median()
    ],
    'Std Dev': [
        results_with_embeddings['f1'].std(),
        results_with_embeddings['bert_f1'].std(),
        results_with_embeddings['embedding_similarity'].std()
    ],
    'Min': [
        results_with_embeddings['f1'].min(),
        results_with_embeddings['bert_f1'].min(),
        results_with_embeddings['embedding_similarity'].min()
    ],
    'Max': [
        results_with_embeddings['f1'].max(),
        results_with_embeddings['bert_f1'].max(),
        results_with_embeddings['embedding_similarity'].max()
    ]
})

print("\n📊 COMPREHENSIVE METRIC SUMMARY TABLE:")
print("="*90)
print(summary_table.to_string(index=False))
print("="*90)


📊 COMPREHENSIVE METRIC SUMMARY TABLE:
              Metric     Mean   Median  Std Dev      Min      Max
            Token F1 0.681689 0.705882 0.158643 0.266667 1.000000
        BERTScore F1 0.792258 0.785845 0.071365 0.618470 0.951434
OpenAI Embedding Sim 0.894459 0.912284 0.060823 0.712469 0.988670


### KG evaluation

In [10]:
# Triple-level Evaluation Preprocessing for RDF Turtle (Cell 82)
# - Parse Turtle triples -> normalized lists for graph_matching.py
# - Gold: prefLabel/property mapping; fallback to camel/uppercase split for URIs without labels; stable blank nodes
# - Pred: no prefLabel/property mapping; humanize URIs by splitting camelCase/uppercase; stable blank nodes
# - Keep literals as-is

# -----------------------------------------------------------------------------
# Load property labels (used only for gold normalization)
# -----------------------------------------------------------------------------
prop_csv = Path('in_ontovocabs/properties.csv')
prop_label_by_uri = {}
prop_label_by_local = {}
prop_label_by_local_alnum = {}

with prop_csv.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        uri = row.get('uri', '').strip()
        label = row.get('label', '').strip()
        if not uri or not label:
            continue
        prop_label_by_uri[uri.lower()] = label.lower()
        local = uri.split('#')[-1].split('/')[-1]
        prop_label_by_local[local.lower()] = label.lower()
        prop_label_by_local_alnum[re.sub(r'[^a-z0-9]', '', local.lower())] = label.lower()

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
blank_regex = re.compile(r'^[a-f0-9]{10,}$')


def camel_to_label(name: str) -> str:
    # Insert spaces before uppercase boundaries and common delimiters, lowercased
    name = re.sub(r'[\-_]+', ' ', name)
    name = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', name)
    name = re.sub(r'(?<=[A-Z])(?=[A-Z][a-z])', ' ', name)
    name = re.sub(r'\s+', ' ', name)
    return name.strip().lower()


def local_name(uri: str) -> str:
    if '#' in uri:
        return uri.split('#')[-1]
    if '/' in uri:
        return uri.rstrip('/').split('/')[-1]
    return uri


def map_predicate_label(pred_term) -> str:
    """Map predicate URI/prefixed/local to human-readable label (gold only)."""
    if isinstance(pred_term, URIRef):
        key_uri = str(pred_term).strip().lower()
        local_part = local_name(str(pred_term))
    else:  # fallback string
        key_uri = str(pred_term).strip().lower()
        if ':' in key_uri and not key_uri.startswith('http'):
            local_part = key_uri.split(':', 1)[1]
        else:
            local_part = local_name(key_uri)
    key_alnum = re.sub(r'[^a-z0-9]', '', local_part.lower())
    local_lower = local_part.lower()

    if key_uri in prop_label_by_uri:
        return prop_label_by_uri[key_uri]
    if local_lower in prop_label_by_local:
        return prop_label_by_local[local_lower]
    if key_alnum in prop_label_by_local_alnum:
        return prop_label_by_local_alnum[key_alnum]
    return camel_to_label(local_part)


# -----------------------------------------------------------------------------
# Gold normalization (prefLabel if available, else camel/uppercase split)
# -----------------------------------------------------------------------------
def normalize_graph_from_turtle_gold(turtle_str: str):
    """Parse Turtle and normalize triples with prefLabel/property mapping (gold)."""
    if not turtle_str or not turtle_str.strip():
        return []
    g = Graph()
    try:
        g.parse(data=turtle_str, format='turtle')
    except Exception:
        return []

    # Collect skos:prefLabel for URI nodes (preserve casing)
    pref_label_map = {}
    for s, _, lbl in g.triples((None, SKOS.prefLabel, None)):
        if isinstance(s, URIRef) and isinstance(lbl, Literal):
            text = str(lbl).strip()
            if text:
                pref_label_map[s] = text

    out = []
    blank_map = {}
    blank_counter = 0

    def canon_node(term):
        nonlocal blank_counter
        if isinstance(term, BNode):
            key = str(term)
            if key not in blank_map:
                blank_counter += 1
                blank_map[key] = f"<blank{blank_counter}>"
            return blank_map[key]
        if isinstance(term, URIRef):
            if term in pref_label_map:
                return pref_label_map[term]
            # fallback: humanize URI
            return camel_to_label(local_name(str(term)))
        if isinstance(term, Literal):
            return str(term).strip()
        return str(term).strip()

    for s, p, o in g:
        s_norm = canon_node(s)
        o_norm = canon_node(o)
        p_norm = map_predicate_label(p)
        out.append([s_norm, p_norm, o_norm])
    return out


# -----------------------------------------------------------------------------
# Pred normalization (no prefLabel; camel/uppercase split for URIs)
# -----------------------------------------------------------------------------
def normalize_graph_from_turtle_pred(turtle_str: str):
    """Parse Turtle and normalize triples; humanize URIs by camel/uppercase splitting (pred)."""
    if not turtle_str or not turtle_str.strip():
        return []
    g = Graph()
    try:
        g.parse(data=turtle_str, format='turtle')
    except Exception:
        return []

    out = []
    blank_map = {}
    blank_counter = 0

    def canon_node(term):
        nonlocal blank_counter
        if isinstance(term, BNode):
            key = str(term)
            if key not in blank_map:
                blank_counter += 1
                blank_map[key] = f"<blank{blank_counter}>"
            return blank_map[key]
        if isinstance(term, URIRef):
            return camel_to_label(local_name(str(term)))
        if isinstance(term, Literal):
            return str(term).strip()
        return str(term).strip()

    for s, p, o in g:
        s_norm = canon_node(s)
        o_norm = canon_node(o)
        p_norm = camel_to_label(local_name(str(p)))
        out.append([s_norm, p_norm, o_norm])
    return out

In [11]:
# Full-batch metrics over all paragraphs (GED calculated separately in cell 76)
import eval_graphs.graph_matching as gm
importlib.reload(gm)
from eval_graphs.graph_matching import (
    get_triple_match_prf,
    get_triple_match_accuracy,
    get_tokens,
    get_bleu_rouge,
    get_bert_score,
    split_to_edges,
    modify_graph,
)

PRED_PATH = 'eval_graphs/RDF_LLMs_raw.json'
GOLD_PATH = 'benchmarks/text_RDF_gs.json'

with open(PRED_PATH, 'r', encoding='utf-8') as f:
    pred_json = json.load(f)
with open(GOLD_PATH, 'r', encoding='utf-8') as f:
    gold_json = json.load(f)

pred_entries = pred_json.get('generated_RDF', [])
gold_entries = gold_json.get('dataset', [])

gold_graphs = []
pred_graphs = []
skipped_no_pred = 0
processed = 0

for p, g in zip(pred_entries, gold_entries):
    turtle_pred = p.get('zero_shot', '')
    turtle_gold = g.get('rdf_graph_turtle', '')
    if not turtle_pred or not turtle_pred.strip():
        skipped_no_pred += 1
        continue
    norm_pred = normalize_graph_from_turtle_pred(turtle_pred)
    norm_gold = normalize_graph_from_turtle_gold(turtle_gold)
    if not norm_pred:  # treat as missing prediction
        skipped_no_pred += 1
        continue
    pred_graphs.append(norm_pred)
    gold_graphs.append(norm_gold)
    processed += 1

print(f"Total pairs: {len(pred_entries)} | evaluated: {processed} | skipped (no pred triples): {skipped_no_pred}")

if not pred_graphs:
    print("No valid predicted graphs to evaluate.")
else:
    print("\n=== Computing metrics (printing as each completes) ===\n")
    
    # 1. Triple-level metrics (micro): Precision, Recall, F1
    print("Computing Triple Match P/R/F1...")
    triple_p, triple_r, triple_f1 = get_triple_match_prf(gold_graphs, pred_graphs)
    print(f"  Triple Match Precision/Recall/F1: {triple_p:.4f} / {triple_r:.4f} / {triple_f1:.4f}\n")

    # 2. Micro triple accuracy across all triples
    print("Computing Triple Match Accuracy...")
    matches = 0
    total_pred = 0
    for pred, gold in zip(pred_graphs, gold_graphs):
        pred_norm = [tuple(t) for t in modify_graph(pred)]
        gold_norm = set(tuple(t) for t in modify_graph(gold))
        matches += sum(1 for t in pred_norm if t in gold_norm)
        total_pred += len(pred_norm)
    triple_acc = matches / total_pred if total_pred else 0.0
    print(f"  Triple Match Accuracy: {triple_acc:.4f}\n")

    # 3. Prepare edges for BLEU/ROUGE/BERTScore
    print("Preparing edges and tokens...")
    gold_edges = split_to_edges(gold_graphs)
    pred_edges = split_to_edges(pred_graphs)
    gold_tokens, pred_tokens = get_tokens(gold_edges, pred_edges)
    
    gold_edge_counts = [len(edges) for edges in gold_edges]
    pred_edge_counts = [len(edges) for edges in pred_edges]
    total_gold_edges = sum(gold_edge_counts)
    total_pred_edges = sum(pred_edge_counts)

    def micro_prf(p_arr, r_arr):
        micro_p = (
            sum(p * c for p, c in zip(p_arr, pred_edge_counts)) / total_pred_edges
            if total_pred_edges else 0.0
        )
        micro_r = (
            sum(r * c for r, c in zip(r_arr, gold_edge_counts)) / total_gold_edges
            if total_gold_edges else 0.0
        )
        micro_f = (2 * micro_p * micro_r / (micro_p + micro_r)) if (micro_p + micro_r) else 0.0
        return micro_p, micro_r, micro_f

    # 4. BLEU and ROUGE
    print("Computing G-BLEU and G-Rouge...")
    prec_r, rec_r, f1_r, prec_b, rec_b, f1_b = get_bleu_rouge(
        gold_tokens, pred_tokens, gold_edges, pred_edges
    )
    rouge_p, rouge_r, rouge_f = micro_prf(prec_r, rec_r)
    bleu_p, bleu_r, bleu_f = micro_prf(prec_b, rec_b)
    print(f"  G-Rouge Precision/Recall/F1: {rouge_p:.4f} / {rouge_r:.4f} / {rouge_f:.4f}")
    print(f"  G-BLEU Precision/Recall/F1: {bleu_p:.4f} / {bleu_r:.4f} / {bleu_f:.4f}\n")

    # 5. BERTScore
    print("Computing BERTScore (this may take a while)...")
    bert_p, bert_r, bert_f = get_bert_score(gold_edges, pred_edges)
    bert_p_m, bert_r_m, bert_f_m = micro_prf(bert_p, bert_r)
    print(f"  BERTScore Precision/Recall/F1: {bert_p_m:.4f} / {bert_r_m:.4f} / {bert_f_m:.4f}\n")

    print("=" * 50)
    print("All metrics computed.")

Total pairs: 105 | evaluated: 105 | skipped (no pred triples): 0

=== Computing metrics (printing as each completes) ===

Computing Triple Match P/R/F1...
  Triple Match Precision/Recall/F1: 0.2465 / 0.2548 / 0.2506

Computing Triple Match Accuracy...
  Triple Match Accuracy: 0.2462

Preparing edges and tokens...
Computing G-BLEU and G-Rouge...
  G-Rouge Precision/Recall/F1: 0.6478 / 0.6703 / 0.6588
  G-BLEU Precision/Recall/F1: 0.6913 / 0.7153 / 0.7031

Computing BERTScore (this may take a while)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Computed bert scores for all pairs
  BERTScore Precision/Recall/F1: 0.8456 / 0.8749 / 0.8600

All metrics computed.


In [12]:
# Calculate GED for ALL graph pairs using iterative optimization with early stopping
import eval_graphs.graph_matching as gm
importlib.reload(gm)
from eval_graphs.graph_matching import get_ged

print(f"Computing GED for {len(gold_graphs)} graph pairs...")
print("Settings: check_interval=10s, stable_threshold=10, max_time=300s")
print("=" * 70)

ged_scores = []
failed_indices = []

for idx, (gold, pred) in enumerate(zip(gold_graphs, pred_graphs)):
    print(f"\n[Pair {idx+1}/{len(gold_graphs)}] Gold: {len(gold)} triples, Pred: {len(pred)} triples")
    
    try:
        ged = get_ged(
            gold, 
            pred, 
            check_interval=10.0,     # Check every 10 seconds
            stable_threshold=10,       # Stop after 10 unchanged checks
            max_time=300.0,           # Max 5 minutes per pair
            verbose=True              # Show progress
        )
        ged_scores.append(ged)
        print(f"  => GED = {ged:.4f}")
    except Exception as e:
        print(f"  => FAILED: {e}")
        failed_indices.append(idx)

print("\n" + "=" * 70)
print("=== GED EVALUATION COMPLETE ===")
print("=" * 70)

if ged_scores:
    ged_array = np.array(ged_scores)
    print(f"\nSuccessfully computed: {len(ged_scores)}/{len(gold_graphs)} pairs")
    print(f"Failed: {len(failed_indices)} pairs")
    if failed_indices:
        print(f"  Failed indices: {failed_indices}")
    
    print(f"\n--- GED Statistics ---")
    print(f"Mean GED:   {ged_array.mean():.4f}")
    print(f"Std GED:    {ged_array.std():.4f}")
    print(f"Min GED:    {ged_array.min():.4f}")
    print(f"Max GED:    {ged_array.max():.4f}")
    print(f"Median GED: {np.median(ged_array):.4f}")
else:
    print("\nNo GED scores computed successfully.")

Computing GED for 105 graph pairs...
Settings: check_interval=10s, stable_threshold=10, max_time=300s

[Pair 1/105] Gold: 51 triples, Pred: 46 triples
[GED] Graph sizes: g1(22 nodes, 51 edges), g2(26 nodes, 46 edges)
[GED] Starting iterative optimization (check every 10.0s, stable threshold=10, max time=300.0s)
[GED] Check #1 at 10.0s: GED=54.0, Stable count=0/10
[GED] Check #2 at 20.0s: GED=54.0, Stable count=0/10
[GED] Check #3 at 30.0s: GED=54.0, Stable count=1/10
[GED] Check #4 at 40.0s: GED=54.0, Stable count=2/10
[GED] Check #5 at 50.0s: GED=54.0, Stable count=3/10
[GED] Check #6 at 60.0s: GED=52.0, Stable count=4/10
[GED] Check #7 at 70.0s: GED=52.0, Stable count=0/10
[GED] Check #8 at 80.0s: GED=52.0, Stable count=1/10
[GED] Check #9 at 90.0s: GED=52.0, Stable count=2/10
[GED] Check #10 at 100.0s: GED=52.0, Stable count=3/10
[GED] Check #11 at 110.0s: GED=51.0, Stable count=4/10
[GED] Check #12 at 120.0s: GED=51.0, Stable count=0/10
[GED] Check #13 at 130.0s: GED=51.0, Stable c